In [7]:
import pandas as pd
import numpy as np

# Load data

In [8]:
neuro_history = pd.read_csv("../data/raw/Locked.PFD.NeurologicalExamination.csv")
followup = pd.read_csv("../data/raw/Locked.PFD.FollowupNeuroExam5.13.21_251001.csv")
site_alloc = pd.read_excel("../data/raw/Site Allocation.xlsx")
# Rename to ensure consistent naming with Site
site_alloc.columns=['Site', 'SiteName', 'PFD', 'PFDExtension', 'Allocation', 'PFDIRB']
treatment = pd.read_csv("../data/raw/Locked.PFD.5.13.21.Treatment.csv")


# --- Step 1: Identify Pre-op Neurological exams ---

In [9]:
# other variables: 2 is not documented
# deep tendon reflexes: 0 is not documented

def check_neuro_signs_presence(row):
    presence = {}

    presence["Papilledema"] = row.get("Papilledema") == 0
    presence["Nystagmus"] = row.get("Nystagmus") == 1
    presence["DisconjugateGaze"] = row.get("DisconjugateGaze") == 1
    presence["ExtraocularPalsies"] = row.get("ExtraocularPalsies") == 1
    presence["FacialWeakness"] = row.get("FacialWeakness") == 1
    presence["DisturbanceInFacialSensation"] = row.get("DisturbanceInFacialSensation") == 1
    presence["HearingLoss"] = row.get("HearingLoss") == 1
    presence["Hoarseness"] = row.get("Hoarseness") == 1
    presence["TongueDeviation"] = row.get("TongueDeviation") == 1
    presence["WeakShrug"] = row.get("WeakShrug") == 1

    presence["LeftUpperExtremityWeak"] = row.get("LeftUpperExtremityStrength") == 0
    presence["RightUpperExtremityWeak"] = row.get("RightUpperExtremityStrength") == 0
    presence["LeftLowerExtremityWeak"] = row.get("LeftLowerExtremityStrength") == 0
    presence["RightLowerExtremityWeak"] = row.get("RightLowerExtremityStrength") == 0

    presence["LighttouchEntireBody"] = row.get("LighttouchEntireBody") == 1
    presence["DLTLeftUpperExtremities"] = row.get("DLTLeftUpperExtremities") == 1
    presence["DLTRightUpperExtremities"] = row.get("DLTRightUpperExtremities") == 1
    presence["DLTLeftLowerExtremities"] = row.get("DLTLeftLowerExtremities") == 1
    presence["DLTRightLowerExtremities"] = row.get("DLTRightLowerExtremities") == 1

    presence["DeficitToPinprickUpperLeft"] = row.get("DeficitToPinprickUpperLeft") == 1
    presence["DeficitToPinprickUpperRight"] = row.get("DeficitToPinprickUpperRight") == 1
    presence["DeficittoPinprickLowerLeft"] = row.get("DeficittoPinprickLowerLeft") == 1
    presence["DeficittoPinprickLowerRight"] = row.get("DeficittoPinprickLowerRight") == 1

    presence["LightTouchTorso"] = row.get("LightTouchTorso") == 1
    presence["LightTouchSaddle"] = row.get("LightTouchSaddle") == 1
    presence["DeepTendonReflexes"] = row.get("DeepTendonReflexes") in [1, 2, 4]
    presence["AnkleClonus"] = row.get("AnkleClonus") == 1
    presence["GaitInstability"] = row.get("GaitInstability") == 1

    return pd.Series(presence)


preop_neuro = neuro_history[['SubjectId']].copy()
neuro_presence = neuro_history.apply(check_neuro_signs_presence, axis=1)
preop_neuro = pd.concat([preop_neuro, neuro_presence], axis=1)

# baseline preop neuro for each patient

In [10]:
preop_neuro
preop_neuro = preop_neuro.astype(int)
preop_neuro

,SubjectId,Papilledema,Nystagmus,DisconjugateGaze,ExtraocularPalsies,FacialWeakness,DisturbanceInFacialSensation,HearingLoss,Hoarseness,TongueDeviation,...,DLTRightLowerExtremities,DeficitToPinprickUpperLeft,DeficitToPinprickUpperRight,DeficittoPinprickLowerLeft,DeficittoPinprickLowerRight,LightTouchTorso,LightTouchSaddle,DeepTendonReflexes,AnkleClonus,GaitInstability
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158,151,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
159,152,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
160,163,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
161,59,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [11]:
list(preop_neuro)

['SubjectId',
 'Papilledema',
 'Nystagmus',
 'DisconjugateGaze',
 'ExtraocularPalsies',
 'FacialWeakness',
 'DisturbanceInFacialSensation',
 'HearingLoss',
 'Hoarseness',
 'TongueDeviation',
 'WeakShrug',
 'LeftUpperExtremityWeak',
 'RightUpperExtremityWeak',
 'LeftLowerExtremityWeak',
 'RightLowerExtremityWeak',
 'LighttouchEntireBody',
 'DLTLeftUpperExtremities',
 'DLTRightUpperExtremities',
 'DLTLeftLowerExtremities',
 'DLTRightLowerExtremities',
 'DeficitToPinprickUpperLeft',
 'DeficitToPinprickUpperRight',
 'DeficittoPinprickLowerLeft',
 'DeficittoPinprickLowerRight',
 'LightTouchTorso',
 'LightTouchSaddle',
 'DeepTendonReflexes',
 'AnkleClonus',
 'GaitInstability']

# --- Step 2: Map follow-up neurological exams ---

In [12]:
neuro_sign_map = {
    "Papilledema": "Papilledema",
    "Nystagmus": "Nystagmus",
    "DisconjugateGaze": "DisconjugateGaze",
    "ExtraocularPalsies": "DocExtraOcPalsies",
    "FacialWeakness": "FacialWeakness",
    "DisturbanceInFacialSensation": "DisturbanceFacSen",
    "HearingLoss": "HearingLoss",
    "Hoarseness": "Hoarseness",
    "TongueDeviation": "TongueDeviation",
    "WeakShrug": "WeakShrug",

    "LeftUpperExtremityWeak": "LeftUpperExtremityStrength",
    "RightUpperExtremityWeak": "RightUpperExtremityStrength",
    "LeftLowerExtremityWeak": "LeftLowerExtremityStrength",
    "RightLowerExtremityWeak": "RightLowerExtremityStrength",

    "LighttouchEntireBody": "DeficitToLightTouchEntireBody",
    "DLTLeftUpperExtremities": "LightTouchSensoryDeficitLUE",
    "DLTRightUpperExtremities": "LightTouchSensoryDeficitRUE",
    "DLTLeftLowerExtremities": "LightTouchSensoryDeficitLLE",
    "DLTRightLowerExtremities": "LightTouchSensoryDeficitRLE",

    "DeficitToPinprickUpperLeft": "LUEDeficitPinprick",
    "DeficitToPinprickUpperRight": "RUEDeficitPinprick",
    "DeficittoPinprickLowerLeft": "LLEDeficitPinprick",
    "DeficittoPinprickLowerRight": "RLEDeficitPinprick",

    "LightTouchTorso": "DeficitLightTouchTorso",
    "LightTouchSaddle": "DeficitLIghtTouchSaddle",  

    "DeepTendonReflexes": "DeepTendonReflexes",
    "AnkleClonus": "FUAnkleClonus",
    "GaitInstability": "GaitStatus"
}



# Make sure treatment date is datetime format
treatment['ChiariSurgeryDate'] = pd.to_datetime(treatment['ChiariSurgeryDate'])

# Merge treatment date into followup
followup = followup.merge(treatment[['SubjectId', 'ChiariSurgeryDate']], on='SubjectId', how='outer')

followup['FollowupDate']  = followup['ExamDate']
followup['FollowupDate'] = pd.to_datetime(followup['FollowupDate'])
followup['DaysPostOp'] = (followup['FollowupDate'] - followup['ChiariSurgeryDate']).dt.days

# Filter to desired range
filtered_followup = followup[(followup['DaysPostOp'] >= 274) & (followup['DaysPostOp'] <= 729)]


In [13]:
for col in preop_neuro.columns:
    if col != 'SubjectId':
        print(col,sum(preop_neuro[col]))

Papilledema 2
Nystagmus 9
DisconjugateGaze 2
ExtraocularPalsies 4
FacialWeakness 0
DisturbanceInFacialSensation 1
HearingLoss 2
Hoarseness 3
TongueDeviation 0
WeakShrug 0
LeftUpperExtremityWeak 1
RightUpperExtremityWeak 3
LeftLowerExtremityWeak 0
RightLowerExtremityWeak 0
LighttouchEntireBody 0
DLTLeftUpperExtremities 3
DLTRightUpperExtremities 3
DLTLeftLowerExtremities 0
DLTRightLowerExtremities 0
DeficitToPinprickUpperLeft 0
DeficitToPinprickUpperRight 0
DeficittoPinprickLowerLeft 0
DeficittoPinprickLowerRight 0
LightTouchTorso 0
LightTouchSaddle 0
DeepTendonReflexes 8
AnkleClonus 3
GaitInstability 13


In [14]:
list(filtered_followup)

['Site',
 'Subject',
 'SubjectId',
 'EncounterId',
 'EncounterDate',
 'ExamDate',
 'Papilledema',
 'Nystagmus',
 'DisconjugateGaze',
 'DocExtraOcPalsies',
 'FacialWeakness',
 'NewFacialWeakness',
 'NewFacialWeaknessSpecify',
 'DisturbanceFacSen',
 'NewDisturbFacialSensation',
 'NewDisturbFacSensatSpecify',
 'HearingLoss',
 'NewHearingLoss',
 'NewHearingLossSpecify',
 'Hoarseness',
 'NewHoarseness',
 'TongueDeviation',
 'NewTongueDeviation',
 'NewTongueDeviationSpecify',
 'WeakShrug',
 'NewWeakShrug',
 'WeakNeckRotation',
 'NewWeakNeckRotation',
 'StrengthStatusDocumented',
 'LeftUpperExtremityStrength',
 'NewLUEStrength',
 'RightUpperExtremityStrength',
 'NewRUEStrength',
 'LeftLowerExtremityStrength',
 'NewLLEStrength',
 'RightLowerExtremityStrength',
 'NewRLEStrength',
 'AssessToLightTouchDocument',
 'DeficitToLightTouchEntireBody',
 'NewDeficitLTEntireBody',
 'DeficitLightTouchTorso',
 'NewDeficitLightTouchTorso',
 'DeficitLIghtTouchSaddle',
 'NewDeficitLightTouchSaddle',
 'DeficitT

In [15]:
# Define helper function to check if a value is "unknown"
def is_unknown(symptom_key, value):
    return value == 5

# Sort follow-ups
filtered_followup = filtered_followup.sort_values(['SubjectId', 'FollowupDate'])
special_mapping = {1: 1, 0: 2, 2: 3, 3: 4, 4: 0}

special_vars = [
    "HearingLoss", "Tongue_Deviation", "Weak Shrug",
    "LeftUpperExtremityStrength", "RightUpperExtremityStrength",
    "LeftLowerExtremityStrength", "RightLowerExtremityStrength"
]

for col in special_vars:
    if col in filtered_followup.columns:
        filtered_followup[col] = filtered_followup[col].replace(special_mapping)

# decide to remove this encoding after discussed with Thanda
# one strange value in fuankleclous
# filtered_followup['FUAnkleClonus'] = (
#     filtered_followup['FUAnkleClonus']
#     .astype(str)
#     .replace({
#         '0': 1,
#         '1': 3,
#         '2': 0,
#     })
# )

# filtered_followup['FUAnkleClonus'] = filtered_followup['FUAnkleClonus'].fillna(0)

# DeepTendonReflexes-2=Resolved, if pre and post values were the same (1, 0 or 3) 
# stable, If pre value was 0 or 3 and post was 1
# Worse, If pre was 1 and post was 0 or, 2, or, 3 then = Improved  
# 4=Not Documented 

# check the data that there is only one 2 in the followup, which works good for this data distribution. need to modify with 0,1,3
# filtered_followup['DeepTendonReflexes'] = filtered_followup['DeepTendonReflexes'].replace({
#     4: 0,
#     2:1
# })

# Papilledema: 2: not documented, 1: absent, 0: present
# after mapping: 2:np.nan, 1:present, 0:absent
#filtered_followup['Papilledema'] = filtered_followup['Papilledema'].replace({2:np.nan,1:0,0:1})

# index_Papilledema: 1 is absent, 0 is present
# filtered_followup['Papilledema'] = filtered_followup['Papilledema'].replace({2:5})

# Initialize results
result_rows = []

# Loop by subject
for subject_id, group in filtered_followup.groupby('SubjectId'):
    result_row = {"SubjectId": subject_id}
    
    for preop_neuro_feature, followup_neuro in neuro_sign_map.items():
        neuro_series = group[followup_neuro]
        
        # if there is no records at all
        if not pd.notna(neuro_series).sum():
            result_row[followup_neuro] = 0
        else:
            # Reverse list of non-NaN values as default sorting is from small to large, early to late
            values = neuro_series.dropna().tolist()[::-1]  # latest first
            last_valid = None
            for val in values:
                # if we find any not "Unknown values"
                if not is_unknown(followup_neuro, val):
                    last_valid = val
                    break
            # if we fail to do so, it means we encounter 0 in the records
            result_row[followup_neuro] = last_valid if last_valid is not None else None
    
    result_rows.append(result_row)
# Convert to DataFrame
latest_followup_df = pd.DataFrame(result_rows)

# merge with site
if 'Site' in followup.columns:
    site_map = followup[['SubjectId', 'Site']].drop_duplicates()
    latest_followup_df = latest_followup_df.merge(site_map, on='SubjectId', how='outer')
latest_followup_df

,SubjectId,Papilledema,Nystagmus,DisconjugateGaze,DocExtraOcPalsies,FacialWeakness,DisturbanceFacSen,HearingLoss,Hoarseness,TongueDeviation,...,LUEDeficitPinprick,RUEDeficitPinprick,LLEDeficitPinprick,RLEDeficitPinprick,DeficitLightTouchTorso,DeficitLIghtTouchSaddle,DeepTendonReflexes,FUAnkleClonus,GaitStatus,Site
0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
1,2,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,1.0,2.0,0.0,1.0,1.0
2,3,2.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,4.0,...,0.0,0.0,0.0,0.0,1.0,1.0,4.0,2.0,4.0,33.0
3,4,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,4.0,...,0.0,0.0,0.0,0.0,1.0,1.0,2.0,2.0,1.0,33.0
4,5,2.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,4.0,2.0,4.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158,164,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.0
159,165,2.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,1.0,4.0,2.0,1.0,28.0
160,166,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0
161,167,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,4.0,7.0


In [16]:
latest_followup_df['DeepTendonReflexes'].value_counts()

DeepTendonReflexes
2.0    92
4.0    39
3.0     2
0.0     1
1.0     1
Name: count, dtype: int64

In [17]:
latest_followup_df[latest_followup_df['SubjectId']==14]

,SubjectId,Papilledema,Nystagmus,DisconjugateGaze,DocExtraOcPalsies,FacialWeakness,DisturbanceFacSen,HearingLoss,Hoarseness,TongueDeviation,...,LUEDeficitPinprick,RUEDeficitPinprick,LLEDeficitPinprick,RLEDeficitPinprick,DeficitLightTouchTorso,DeficitLIghtTouchSaddle,DeepTendonReflexes,FUAnkleClonus,GaitStatus,Site
12,14,2.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,1.0,4.0,2.0,1.0,1.0


In [18]:
latest_followup_df['Papilledema'].value_counts()

Papilledema
2.0    88
1.0    47
Name: count, dtype: int64

In [19]:
latest_followup_df['FUAnkleClonus'].value_counts()

FUAnkleClonus
2.0    88
0.0    46
1.0     1
Name: count, dtype: int64

In [20]:
latest_followup_df['FUAnkleClonus'].value_counts()

FUAnkleClonus
2.0    88
0.0    46
1.0     1
Name: count, dtype: int64

In [21]:
latest_followup_df = latest_followup_df.rename(
    columns={col: f"Index_{col}" for col in latest_followup_df.columns if col not in ["SubjectId", "Site"]}
)

In [22]:
list(latest_followup_df)

['SubjectId',
 'Index_Papilledema',
 'Index_Nystagmus',
 'Index_DisconjugateGaze',
 'Index_DocExtraOcPalsies',
 'Index_FacialWeakness',
 'Index_DisturbanceFacSen',
 'Index_HearingLoss',
 'Index_Hoarseness',
 'Index_TongueDeviation',
 'Index_WeakShrug',
 'Index_LeftUpperExtremityStrength',
 'Index_RightUpperExtremityStrength',
 'Index_LeftLowerExtremityStrength',
 'Index_RightLowerExtremityStrength',
 'Index_DeficitToLightTouchEntireBody',
 'Index_LightTouchSensoryDeficitLUE',
 'Index_LightTouchSensoryDeficitRUE',
 'Index_LightTouchSensoryDeficitLLE',
 'Index_LightTouchSensoryDeficitRLE',
 'Index_LUEDeficitPinprick',
 'Index_RUEDeficitPinprick',
 'Index_LLEDeficitPinprick',
 'Index_RLEDeficitPinprick',
 'Index_DeficitLightTouchTorso',
 'Index_DeficitLIghtTouchSaddle',
 'Index_DeepTendonReflexes',
 'Index_FUAnkleClonus',
 'Index_GaitStatus',
 'Site']

In [23]:
# latest_followup_df = latest_followup_df.replace({0: 5})
# latest_followup_df = latest_followup_df.fillna(5)

In [24]:
preop_neuro = preop_neuro.rename(
    columns={col: f"Preop_{col}" for col in preop_neuro.columns if col != "SubjectId"}
)
preop_neuro

,SubjectId,Preop_Papilledema,Preop_Nystagmus,Preop_DisconjugateGaze,Preop_ExtraocularPalsies,Preop_FacialWeakness,Preop_DisturbanceInFacialSensation,Preop_HearingLoss,Preop_Hoarseness,Preop_TongueDeviation,...,Preop_DLTRightLowerExtremities,Preop_DeficitToPinprickUpperLeft,Preop_DeficitToPinprickUpperRight,Preop_DeficittoPinprickLowerLeft,Preop_DeficittoPinprickLowerRight,Preop_LightTouchTorso,Preop_LightTouchSaddle,Preop_DeepTendonReflexes,Preop_AnkleClonus,Preop_GaitInstability
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158,151,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
159,152,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
160,163,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
161,59,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [25]:
# Merge pre-op and post-op data
merged = pd.merge(preop_neuro, latest_followup_df, on='SubjectId', how='outer')
neuro_cols = neuro_sign_map.keys()
# Filter only patients who had any True in pre-op neuros
neuro_cols_preop = [f"Preop_{col}" for col in neuro_cols]
merged['AnyPreopNeuro'] = merged[neuro_cols_preop].any(axis=1)
# baseline_subjects_df = merged[merged['AnyPreopNeuro'] == True].drop(columns=['AnyPreopNeuro'])
baseline_subjects_df = merged.drop(columns=['AnyPreopNeuro'])
# Final per-subject output
subject_results_df = baseline_subjects_df.copy()


In [26]:
subject_results_df['Preop_DeepTendonReflexes'].value_counts()

Preop_DeepTendonReflexes
0    155
1      8
Name: count, dtype: int64

In [27]:
subject_results_df['Index_DeepTendonReflexes'].value_counts()

Index_DeepTendonReflexes
2.0    92
4.0    39
3.0     2
0.0     1
1.0     1
Name: count, dtype: int64

In [28]:

index_cols = [col for col in subject_results_df.columns if col.startswith("Index")]

has_zero = (subject_results_df[index_cols] == 0).any().any()

print("Any zeros in Index columns?", has_zero)

zero_mask = (subject_results_df[index_cols] == 0)
rows_with_zero = subject_results_df[zero_mask.any(axis=1)]

print("Rows with at least one zero in Index columns:")
print(rows_with_zero)


Any zeros in Index columns? True
Rows with at least one zero in Index columns:
     SubjectId  Preop_Papilledema  Preop_Nystagmus  Preop_DisconjugateGaze  \
1            2                  0                0                       0   
2            3                  0                0                       0   
3            4                  0                0                       0   
4            5                  0                0                       0   
5            6                  0                0                       0   
..         ...                ...              ...                     ...   
153        159                  0                0                       0   
157        163                  0                0                       0   
159        165                  0                0                       0   
161        167                  0                0                       0   
162        168                  0                0             

In [29]:
subject_results_df

,SubjectId,Preop_Papilledema,Preop_Nystagmus,Preop_DisconjugateGaze,Preop_ExtraocularPalsies,Preop_FacialWeakness,Preop_DisturbanceInFacialSensation,Preop_HearingLoss,Preop_Hoarseness,Preop_TongueDeviation,...,Index_LUEDeficitPinprick,Index_RUEDeficitPinprick,Index_LLEDeficitPinprick,Index_RLEDeficitPinprick,Index_DeficitLightTouchTorso,Index_DeficitLIghtTouchSaddle,Index_DeepTendonReflexes,Index_FUAnkleClonus,Index_GaitStatus,Site
0,1,0,0,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
1,2,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,1.0,1.0,2.0,0.0,1.0,1.0
2,3,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,1.0,1.0,4.0,2.0,4.0,33.0
3,4,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,1.0,1.0,2.0,2.0,1.0,33.0
4,5,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,4.0,2.0,4.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158,164,0,0,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.0
159,165,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,1.0,1.0,4.0,2.0,1.0,28.0
160,166,0,0,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0
161,167,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,4.0,7.0


In [30]:
# Check for missing values in subject_results_df
missing_summary = subject_results_df.isnull().sum()
print("Missing values per column:")
print(missing_summary[missing_summary > 0])

Missing values per column:
Index_Papilledema                      28
Index_Nystagmus                        28
Index_DisconjugateGaze                 28
Index_DocExtraOcPalsies                28
Index_FacialWeakness                   28
Index_DisturbanceFacSen                28
Index_HearingLoss                      28
Index_Hoarseness                       28
Index_TongueDeviation                  28
Index_WeakShrug                        28
Index_LeftUpperExtremityStrength       28
Index_RightUpperExtremityStrength      28
Index_LeftLowerExtremityStrength       28
Index_RightLowerExtremityStrength      28
Index_DeficitToLightTouchEntireBody    28
Index_LightTouchSensoryDeficitLUE      28
Index_LightTouchSensoryDeficitRUE      28
Index_LightTouchSensoryDeficitLLE      28
Index_LightTouchSensoryDeficitRLE      28
Index_LUEDeficitPinprick               28
Index_RUEDeficitPinprick               28
Index_LLEDeficitPinprick               28
Index_RLEDeficitPinprick               28
Index_D

In [31]:
for col in subject_results_df.columns:
    if col != 'SubjectId':
        print(col,subject_results_df[col].value_counts())

Preop_Papilledema Preop_Papilledema
0    161
1      2
Name: count, dtype: int64
Preop_Nystagmus Preop_Nystagmus
0    154
1      9
Name: count, dtype: int64
Preop_DisconjugateGaze Preop_DisconjugateGaze
0    161
1      2
Name: count, dtype: int64
Preop_ExtraocularPalsies Preop_ExtraocularPalsies
0    159
1      4
Name: count, dtype: int64
Preop_FacialWeakness Preop_FacialWeakness
0    163
Name: count, dtype: int64
Preop_DisturbanceInFacialSensation Preop_DisturbanceInFacialSensation
0    162
1      1
Name: count, dtype: int64
Preop_HearingLoss Preop_HearingLoss
0    161
1      2
Name: count, dtype: int64
Preop_Hoarseness Preop_Hoarseness
0    160
1      3
Name: count, dtype: int64
Preop_TongueDeviation Preop_TongueDeviation
0    163
Name: count, dtype: int64
Preop_WeakShrug Preop_WeakShrug
0    163
Name: count, dtype: int64
Preop_LeftUpperExtremityWeak Preop_LeftUpperExtremityWeak
0    162
1      1
Name: count, dtype: int64
Preop_RightUpperExtremityWeak Preop_RightUpperExtremityWeak
0  

In [32]:
preop_to_post_new = {
    "Papilledema": ("Papilledema", None),
    "Nystagmus": ("Nystagmus", None),
    "DisconjugateGaze": ("DisconjugateGaze", None),
    "ExtraocularPalsies": ("DocExtraOcPalsies", None),

    "FacialWeakness": ("FacialWeakness", "NewFacialWeakness"),
    "DisturbanceInFacialSensation": ("DisturbanceFacSen", "NewDisturbFacialSensation"),
    "HearingLoss": ("HearingLoss", "NewHearingLoss"),
    "Hoarseness": ("Hoarseness", "NewHoarseness"),
    "TongueDeviation": ("TongueDeviation", "NewTongueDeviation"),
    "WeakShrug": ("WeakShrug", "NewWeakShrug"),

    "LeftUpperExtremityStrength": ("LeftUpperExtremityStrength", "NewLUEStrength"),
    "RightUpperExtremityStrength": ("RightUpperExtremityStrength", "NewRUEStrength"),
    "LeftLowerExtremityStrength": ("LeftLowerExtremityStrength", "NewLLEStrength"),
    "RightLowerExtremityStrength": ("RightLowerExtremityStrength", "NewRLEStrength"),

    "LighttouchEntireBody": ("DeficitToLightTouchEntireBody", "NewDeficitLTEntireBody"),
    "DLTLeftUpperExtremities": ("LightTouchSensoryDeficitLUE", "NewLightTouchSensoryDefLUE"),
    "DLTRightUpperExtremities": ("LightTouchSensoryDeficitRUE", "NewLightTouchSensoryDefRUE"),
    "DLTLeftLowerExtremities": ("LightTouchSensoryDeficitLLE", "NewLightTouchSensoryDefLLE"),
    "DLTRightLowerExtremities": ("LightTouchSensoryDeficitRLE", "NewLightTouchSensoryDefRLE"),


    "DeficitToPinprickUpperLeft": ("LUEDeficitPinprick", "NewLUEDeficitPinprick"),
    "DeficitToPinprickUpperRight": ("RUEDeficitPinprick", "NewRUEDeficitPinprick"),
    "DeficittoPinprickLowerLeft": ("LLEDeficitPinprick", "NewLLEDeficitPinprick"),
    "DeficittoPinprickLowerRight": ("RLEDeficitPinprick", "NewRLEDeficitPinprick"),

    "LightTouchTorso": ("DeficitLightTouchTorso", "NewDeficitLightTouchTorso"),
    "LightTouchSaddle": ("DeficitLIghtTouchSaddle", "NewDeficitLightTouchSaddle"),

    "DeepTendonReflexes": ("DeepTendonReflexes", "NewDeepTendonReflex"),  
    "AnkleClonus": ("FUAnkleClonus", None),
    "GaitInstability": ("GaitStatus", None),
}


neuro_sign_map = {k: v[0] for k, v in preop_to_post_new.items()}



In [33]:
filtered_followup

,Site,Subject,SubjectId,EncounterId,EncounterDate,ExamDate,Papilledema,Nystagmus,DisconjugateGaze,DocExtraOcPalsies,...,RecordAddDate,FUAnkleClonus,FUBabinskiReflex,FUHoffmansReflex,FURombergResponse,FUDysmetria,Unnamed: 83,ChiariSurgeryDate,FollowupDate,DaysPostOp
7,1.0,8002.0,2,114.0,6/8/2017 0:00,6/8/2017,1.0,1.0,1.0,1.0,...,12:19.5,0.0,2.0,2.0,2.0,2.0,NaN,2016-08-29,2017-06-08,283.0
105,33.0,8001.0,3,137.0,8/10/2017 0:00,8/10/2017,2.0,0.0,1.0,1.0,...,12:32.4,2.0,2.0,2.0,2.0,2.0,NaN,2016-09-07,2017-08-10,337.0
94,33.0,8001.0,3,141.0,9/4/2017 0:00,9/4/2017,2.0,1.0,1.0,1.0,...,17:35.6,2.0,2.0,2.0,2.0,2.0,NaN,2016-09-07,2017-09-04,362.0
83,33.0,8001.0,3,140.0,9/7/2017 0:00,9/7/2017,2.0,1.0,1.0,1.0,...,41:45.6,2.0,2.0,2.0,2.0,2.0,NaN,2016-09-07,2017-09-07,365.0
17,33.0,8001.0,3,142.0,9/19/2017 0:00,9/19/2017,2.0,4.0,0.0,0.0,...,20:47.1,2.0,2.0,2.0,2.0,2.0,NaN,2016-09-07,2017-09-19,377.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2783,49.0,8004.0,159,10782.0,9/18/2019 0:00,9/18/2019,2.0,1.0,1.0,1.0,...,45:58.9,0.0,2.0,2.0,2.0,2.0,NaN,2018-09-26,2019-09-18,357.0
2835,48.0,8002.0,163,10738.0,9/19/2019 0:00,9/19/2019,1.0,1.0,1.0,1.0,...,25:53.9,2.0,2.0,2.0,2.0,2.0,NaN,2018-08-27,2019-09-19,388.0
2877,28.0,8003.0,165,10754.0,11/8/2019 0:00,11/8/2019,2.0,1.0,1.0,0.0,...,32:39.5,2.0,2.0,2.0,2.0,2.0,NaN,2018-12-13,2019-11-08,330.0
2891,7.0,8003.0,167,10777.0,1/31/2020 0:00,1/31/2020,2.0,0.0,0.0,0.0,...,13:34.2,2.0,2.0,2.0,2.0,2.0,NaN,2018-12-11,2020-01-31,416.0


In [34]:
for symtom in preop_to_post_new.values():
    print(symtom)
    for value in symtom:
        if value is not None:
            print(filtered_followup[value].value_counts())

('Papilledema', None)
Papilledema
2.0    116
1.0     61
Name: count, dtype: int64
('Nystagmus', None)
Nystagmus
1.0    146
0.0     27
4.0      2
3.0      2
Name: count, dtype: int64
('DisconjugateGaze', None)
DisconjugateGaze
1.0    159
0.0     15
3.0      2
4.0      1
Name: count, dtype: int64
('DocExtraOcPalsies', None)
DocExtraOcPalsies
1.0    161
0.0     13
3.0      1
Name: count, dtype: int64
('FacialWeakness', 'NewFacialWeakness')
FacialWeakness
1.0    158
0.0     19
Name: count, dtype: int64
NewFacialWeakness
0.0    176
Name: count, dtype: int64
('DisturbanceFacSen', 'NewDisturbFacialSensation')
DisturbanceFacSen
1.0    108
0.0     69
Name: count, dtype: int64
NewDisturbFacialSensation
0.0    177
Name: count, dtype: int64
('HearingLoss', 'NewHearingLoss')
HearingLoss
1.0    117
0.0     60
Name: count, dtype: int64
NewHearingLoss
0.0    177
Name: count, dtype: int64
('Hoarseness', 'NewHoarseness')
Hoarseness
1.0    110
0.0     67
Name: count, dtype: int64
NewHoarseness
0.0    177

In [35]:
def extract_new_reported_symptoms(
    df_followup: pd.DataFrame,
    mapping: dict,                       # {preop_key: (index_col, new_col or None)}
    subject_id_col: str = "SubjectId",
    date_col: str = "FollowupDate",
) -> pd.DataFrame:
    """
    For each pre-op key in mapping, look up its 'new/overall' column (if any),
    and for each subject take the LATEST value that is NOT NaN, 0, or 5.
    If none found, set 5 (unknown).
    Output columns are named New_<preop_key>.
    """
    # sort ascending by date so we can scan from the end (latest first)
    df_sorted = df_followup.sort_values([subject_id_col, date_col], ascending=[True, True])

    def is_unknown(val):
        if pd.isna(val):
            return True
        try:
            v = float(val)
            return v in 5
        except Exception:
            # non-numeric strings are considered valid (keep) unless NaN
            return False

    out_rows = []
    for sid, grp in df_sorted.groupby(subject_id_col, sort=False):
        row = {subject_id_col: sid}

        for preop_key, (_, new_col) in mapping.items():
            out_col = f"New_{preop_key}"

            # If no mapped new/overall column, mark unknown (5)
            if (new_col is None) or (new_col not in grp.columns):
                row[out_col] = 5
                continue

            ser = grp[new_col]

            # Scan from latest to earliest, pick first non-unknown
            last_valid = 5
            # use .iloc[::-1] to iterate from latest
            for val in ser.iloc[::-1]:
                if not is_unknown(val):
                    last_valid = val
                    break

            row[out_col] = last_valid

        out_rows.append(row)

    return pd.DataFrame(out_rows)

new_symptom_flags = extract_new_reported_symptoms(filtered_followup, preop_to_post_new)
new_symptom_flags = new_symptom_flags.drop_duplicates()


In [36]:
new_symptom_flags['New_HearingLoss'].value_counts()

New_HearingLoss
0.0    135
Name: count, dtype: int64

In [37]:
missing_summary = new_symptom_flags.isnull().sum()
print("Missing values per column in new_symptom_flags:")
print(missing_summary[missing_summary > 0])

Missing values per column in new_symptom_flags:
Series([], dtype: int64)


In [38]:
subject_results_df = pd.merge(subject_results_df, new_symptom_flags, on='SubjectId', how='left')
subject_results_df = subject_results_df.fillna(5)
subject_results_df

,SubjectId,Preop_Papilledema,Preop_Nystagmus,Preop_DisconjugateGaze,Preop_ExtraocularPalsies,Preop_FacialWeakness,Preop_DisturbanceInFacialSensation,Preop_HearingLoss,Preop_Hoarseness,Preop_TongueDeviation,...,New_DLTRightLowerExtremities,New_DeficitToPinprickUpperLeft,New_DeficitToPinprickUpperRight,New_DeficittoPinprickLowerLeft,New_DeficittoPinprickLowerRight,New_LightTouchTorso,New_LightTouchSaddle,New_DeepTendonReflexes,New_AnkleClonus,New_GaitInstability
0,1,0,0,0,0,0,0,0,0,0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0
1,2,0,0,0,0,0,0,0,0,0,...,0.0,5.0,5.0,5.0,5.0,0.0,0.0,0.0,5.0,5.0
2,3,0,0,0,0,0,0,0,0,0,...,0.0,5.0,5.0,5.0,5.0,0.0,0.0,0.0,5.0,5.0
3,4,0,0,0,0,0,0,0,0,0,...,0.0,5.0,5.0,5.0,5.0,0.0,0.0,0.0,5.0,5.0
4,5,0,0,0,0,0,0,0,0,0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158,164,0,0,0,0,0,0,0,0,0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0
159,165,0,0,0,0,0,0,0,0,0,...,0.0,5.0,5.0,5.0,5.0,0.0,0.0,0.0,5.0,5.0
160,166,0,0,0,0,0,0,0,0,0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0
161,167,0,0,0,0,0,0,0,0,0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,0.0,5.0,5.0


In [39]:
missing_summary = subject_results_df.isnull().sum()
print("Missing values per column in subject_results_df:")
print(missing_summary[missing_summary > 0])

Missing values per column in subject_results_df:
Series([], dtype: int64)


In [40]:
# Locate the row for SubjectId 82
subject_row = subject_results_df[subject_results_df['SubjectId'] == 82]

# Swap the values of Index_LightTouchSensoryDeficitLUE and Index_LightTouchSensoryDeficitRUE after checking the source file from Thanda
subject_results_df.loc[subject_row.index, ['Index_LightTouchSensoryDeficitLUE', 'Index_LightTouchSensoryDeficitRUE']] = \
    subject_row[['Index_LightTouchSensoryDeficitRUE', 'Index_LightTouchSensoryDeficitLUE']].values

In [41]:
# Create combined extremity strength and DLT features
subject_results_df["Preop_CombinedUpperExtremityWeak"] = (
    (subject_results_df["Preop_LeftUpperExtremityWeak"] == 1) |
    (subject_results_df["Preop_RightUpperExtremityWeak"] == 1)
).astype(int)

subject_results_df["Preop_CombinedLowerExtremityWeak"] = (
    (subject_results_df["Preop_LeftLowerExtremityWeak"] == 1) |
    (subject_results_df["Preop_RightLowerExtremityWeak"] == 1)
).astype(int)

subject_results_df["Preop_CombinedUpperDLTExtremities"] = (
    (subject_results_df["Preop_DLTLeftUpperExtremities"] == 1) |
    (subject_results_df["Preop_DLTRightUpperExtremities"] == 1)
).astype(int)

subject_results_df["Preop_CombinedLowerDLTExtremities"] = (
    (subject_results_df["Preop_DLTLeftLowerExtremities"] == 1) |
    (subject_results_df["Preop_DLTRightLowerExtremities"] == 1)
).astype(int)


# subject_results_df["Index_CombinedUpperExtremityStrength"] = (
#     (subject_results_df["Index_LeftUpperExtremityStrength"] == 1) |
#     (subject_results_df["Index_RightUpperExtremityStrength"] == 1)
# ).astype(int)

# subject_results_df["Index_CombinedLowerExtremityStrength"] = (
#     (subject_results_df["Index_LeftLowerExtremityStrength"] == 1) |
#     (subject_results_df["Index_RightLowerExtremityStrength"] == 1)
# ).astype(int)

# subject_results_df["Index_CombinedUpperDLTExtremity"] = np.where(
#     (subject_results_df["Index_LightTouchSensoryDeficitLUE"] > 0) & (subject_results_df["Index_LightTouchSensoryDeficitRUE"] > 0),
#     np.minimum(subject_results_df["Index_LightTouchSensoryDeficitLUE"], subject_results_df["Index_LightTouchSensoryDeficitRUE"]),
#     subject_results_df[["Index_LightTouchSensoryDeficitLUE", "Index_LightTouchSensoryDeficitRUE"]].max(axis=1)
# )

# # For CombinedLowerDLTExtremity
# subject_results_df["Index_CombinedLowerDLTExtremity"] = np.where(
#     (subject_results_df["Index_LightTouchSensoryDeficitLLE"] > 0) & (subject_results_df["Index_LightTouchSensoryDeficitRLE"] > 0),
#     np.minimum(subject_results_df["Index_LightTouchSensoryDeficitLLE"], subject_results_df["Index_LightTouchSensoryDeficitRLE"]),
#     subject_results_df[["Index_LightTouchSensoryDeficitLLE", "Index_LightTouchSensoryDeficitRLE"]].max(axis=1))


# # Create combined extremity strength and DLT features
# subject_results_df["New_CombinedUpperExtremityStrength"] = (
#     (subject_results_df["New_LeftUpperExtremityStrength"] == 1) |
#     (subject_results_df["New_RightUpperExtremityStrength"] == 1)
# ).astype(int)

# subject_results_df["New_CombinedLowerExtremityStrength"] = (
#     (subject_results_df["New_LeftLowerExtremityStrength"] == 1) |
#     (subject_results_df["New_RightLowerExtremityStrength"] == 1)
# ).astype(int)

# subject_results_df["New_CombinedUpperDLTExtremities"] = (
#     (subject_results_df["New_DLTLeftUpperExtremities"] == 1) |
#     (subject_results_df["New_DLTRightUpperExtremities"] == 1)
# ).astype(int)

# subject_results_df["New_CombinedLowerDLTExtremities"] = (
#     (subject_results_df["New_DLTLeftLowerExtremities"] == 1) |
#     (subject_results_df["New_DLTRightLowerExtremities"] == 1)
# ).astype(int)


In [42]:
combine_groups_strength = {
    "CombinedUpperExtremityStrength": ("Index_LeftUpperExtremityStrength", "Index_RightUpperExtremityStrength"),
    "CombinedLowerExtremityStrength": ("Index_LeftLowerExtremityStrength", "Index_RightLowerExtremityStrength"),
    "CombinedUpperDLTExtremities": ("Index_LightTouchSensoryDeficitLUE", "Index_LightTouchSensoryDeficitRUE"),
    "CombinedLowerDLTExtremities": ("Index_LightTouchSensoryDeficitLLE", "Index_LightTouchSensoryDeficitRLE"),
}

def custom_index_combine_left_right(L, R):
    """
    Hard-encoded combination for INDEX values (1..5) using the exact rules provided.
    Returns np.nan if either side is missing or outside 1..5.
    """
    if R == 1 and L == 1: 
        return 1
    elif R == 1 and L == 2: 
        return 2
    elif R == 1 and L == 3: 
        return 2
    elif R == 1 and L == 4: 
        return 4
    elif R == 1 and L == 5: 
        return 1

    elif R == 2 and L == 1: 
        return 2
    elif R == 2 and L == 2: 
        return 2
    elif R == 2 and L == 3: 
        return 2
    elif R == 2 and L == 4: 
        return 4
    elif R == 2 and L == 5: 
        return 2

    elif R == 3 and L == 1: 
        return 2
    elif R == 3 and L == 2: 
        return 2
    elif R == 3 and L == 3: 
        return 3
    elif R == 3 and L == 4: 
        return 4
    elif R == 3 and L == 5: 
        return 3

    # Right = 4
    elif R == 4 and L == 1: 
        return 4
    elif R == 4 and L == 2: 
        return 4
    elif R == 4 and L == 3: 
        return 4
    elif R == 4 and L == 4: 
        return 4
    elif R == 4 and L == 5: 
        return 4

    # Right = 5
    elif R == 5 and L == 1: 
        return 1
    elif R == 5 and L == 2: 
        return 2
    elif R == 5 and L == 3: 
        return 3
    elif R == 5 and L == 4: 
        return 4
    elif R == 5 and L == 5: 
        return 5

    return 5

# Apply to each pair (left, right)
for feature, (left_col, right_col) in combine_groups_strength.items():
    combined_col = f"Index_{feature}"
    subject_results_df[combined_col] = subject_results_df.apply(
        lambda row: custom_index_combine_left_right(row[left_col], row[right_col]), axis=1
    )
for feature in combine_groups_strength:
    subject_results_df[f"Index_{feature}"] = subject_results_df[f"Index_{feature}"].fillna(5)


for feature, (left_col, right_col) in combine_groups_strength.items():
    print(f"Processing feature: {feature}")
    print(subject_results_df[f"Index_{feature}"].value_counts())


Processing feature: CombinedUpperExtremityStrength
Index_CombinedUpperExtremityStrength
1    117
5     46
Name: count, dtype: int64
Processing feature: CombinedLowerExtremityStrength
Index_CombinedLowerExtremityStrength
1    118
5     45
Name: count, dtype: int64
Processing feature: CombinedUpperDLTExtremities
Index_CombinedUpperDLTExtremities
1    95
5    66
2     2
Name: count, dtype: int64
Processing feature: CombinedLowerDLTExtremities
Index_CombinedLowerDLTExtremities
1    98
5    65
Name: count, dtype: int64


In [43]:
combine_groups_new = {
    "CombinedUpperExtremityStrength": ("New_LeftUpperExtremityStrength", "New_RightUpperExtremityStrength"),
    "CombinedLowerExtremityStrength": ("New_LeftLowerExtremityStrength", "New_RightLowerExtremityStrength"),
    "CombinedUpperDLTExtremities": ("New_DLTLeftUpperExtremities", "New_DLTRightUpperExtremities"),
    "CombinedLowerDLTExtremities": ("New_DLTLeftLowerExtremities", "New_DLTRightLowerExtremities"),
}

def custom_new_combine_left_right(L, R):
    # Right = 1
    if R == 1 and L == 0: 
        return 1
    elif R == 1 and L == 1: 
        return 1
    elif R == 1 and L == 5: 
        return 1

    # Right = 0
    elif R == 0 and L == 0: 
        return 0
    elif R == 0 and L == 1: 
        return 1
    elif R == 0 and L == 5: 
        return 0

    # Right = 5
    elif R == 5 and L == 0: 
        return 0
    elif R == 5 and L == 1: 
        return 1
    elif R == 5 and L == 5: 
        return 5

    # Fallback (e.g., missing/other values) -> 5
    return 5

# Apply to each pair (left, right) to build New_Combined* columns
for feature, (left_col, right_col) in combine_groups_new.items():
    combined_col = f"New_{feature}"
    subject_results_df[combined_col] = subject_results_df.apply(
        lambda row: custom_new_combine_left_right(row[left_col], row[right_col]), axis=1
    )


for feature in combine_groups_new:
    subject_results_df[f"New_{feature}"] = subject_results_df[f"New_{feature}"].fillna(5)


for feature in combine_groups_new:
    print(f"Processing NEW feature: {feature}")
    print(subject_results_df[f"New_{feature}"].value_counts(dropna=False))
    print()


Processing NEW feature: CombinedUpperExtremityStrength
New_CombinedUpperExtremityStrength
0    119
5     44
Name: count, dtype: int64

Processing NEW feature: CombinedLowerExtremityStrength
New_CombinedLowerExtremityStrength
0    118
5     44
1      1
Name: count, dtype: int64

Processing NEW feature: CombinedUpperDLTExtremities
New_CombinedUpperDLTExtremities
0    97
5    65
1     1
Name: count, dtype: int64

Processing NEW feature: CombinedLowerDLTExtremities
New_CombinedLowerDLTExtremities
0    97
5    65
1     1
Name: count, dtype: int64



In [44]:
# Step 1: Ensure column name consistency
site_alloc.columns = site_alloc.columns.str.strip()  # remove leading/trailing spaces
subject_results_df.columns = subject_results_df.columns.str.strip()

# Step 2: Merge on 'Site' key
subject_results_df = pd.merge(subject_results_df, site_alloc[['Site', 'Allocation']], on='Site', how='left')

In [45]:
subject_results_df

,SubjectId,Preop_Papilledema,Preop_Nystagmus,Preop_DisconjugateGaze,Preop_ExtraocularPalsies,Preop_FacialWeakness,Preop_DisturbanceInFacialSensation,Preop_HearingLoss,Preop_Hoarseness,Preop_TongueDeviation,...,Preop_CombinedLowerDLTExtremities,Index_CombinedUpperExtremityStrength,Index_CombinedLowerExtremityStrength,Index_CombinedUpperDLTExtremities,Index_CombinedLowerDLTExtremities,New_CombinedUpperExtremityStrength,New_CombinedLowerExtremityStrength,New_CombinedUpperDLTExtremities,New_CombinedLowerDLTExtremities,Allocation
0,1,0,0,0,0,0,0,0,0,0,...,0,5,5,5,5,5,5,5,5,PFD
1,2,0,0,0,0,0,0,0,0,0,...,0,1,1,1,1,0,0,0,0,PFD
2,3,0,0,0,0,0,0,0,0,0,...,0,1,1,1,1,0,0,0,0,PFDD
3,4,0,0,0,0,0,0,0,0,0,...,0,1,1,1,1,0,0,0,0,PFDD
4,5,0,0,0,0,0,0,0,0,0,...,0,5,5,5,5,5,5,5,5,PFD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158,164,0,0,0,0,0,0,0,0,0,...,0,5,5,5,5,5,5,5,5,PFD
159,165,0,0,0,0,0,0,0,0,0,...,0,1,1,1,1,0,0,0,0,PFDD
160,166,0,0,0,0,0,0,0,0,0,...,0,5,5,5,5,5,5,5,5,PFD
161,167,0,0,0,0,0,0,0,0,0,...,0,5,5,5,5,5,5,5,5,PFD


In [46]:
list(subject_results_df)

['SubjectId',
 'Preop_Papilledema',
 'Preop_Nystagmus',
 'Preop_DisconjugateGaze',
 'Preop_ExtraocularPalsies',
 'Preop_FacialWeakness',
 'Preop_DisturbanceInFacialSensation',
 'Preop_HearingLoss',
 'Preop_Hoarseness',
 'Preop_TongueDeviation',
 'Preop_WeakShrug',
 'Preop_LeftUpperExtremityWeak',
 'Preop_RightUpperExtremityWeak',
 'Preop_LeftLowerExtremityWeak',
 'Preop_RightLowerExtremityWeak',
 'Preop_LighttouchEntireBody',
 'Preop_DLTLeftUpperExtremities',
 'Preop_DLTRightUpperExtremities',
 'Preop_DLTLeftLowerExtremities',
 'Preop_DLTRightLowerExtremities',
 'Preop_DeficitToPinprickUpperLeft',
 'Preop_DeficitToPinprickUpperRight',
 'Preop_DeficittoPinprickLowerLeft',
 'Preop_DeficittoPinprickLowerRight',
 'Preop_LightTouchTorso',
 'Preop_LightTouchSaddle',
 'Preop_DeepTendonReflexes',
 'Preop_AnkleClonus',
 'Preop_GaitInstability',
 'Index_Papilledema',
 'Index_Nystagmus',
 'Index_DisconjugateGaze',
 'Index_DocExtraOcPalsies',
 'Index_FacialWeakness',
 'Index_DisturbanceFacSen',
 '

In [47]:
subject_results_df['New_HearingLoss'].value_counts()

New_HearingLoss
0.0    135
5.0     28
Name: count, dtype: int64

In [48]:
ordered_cols = [
    "SubjectId",

    "Preop_Papilledema", "Index_Papilledema",
    "Preop_Nystagmus", "Index_Nystagmus",
    "Preop_DisconjugateGaze", "Index_DisconjugateGaze",
    "Preop_ExtraocularPalsies", "Index_DocExtraOcPalsies",
    "Preop_FacialWeakness", "Index_FacialWeakness", "New_FacialWeakness",
    "Preop_DisturbanceInFacialSensation", "Index_DisturbanceFacSen", "New_DisturbanceInFacialSensation",
    "Preop_HearingLoss", "Index_HearingLoss", "New_HearingLoss",
    "Preop_Hoarseness", "Index_Hoarseness", "New_Hoarseness",
    "Preop_TongueDeviation", "Index_TongueDeviation", "New_TongueDeviation",
    "Preop_WeakShrug", "Index_WeakShrug", "New_WeakShrug",

    "Preop_CombinedUpperExtremityWeak","Index_CombinedUpperExtremityStrength", "New_CombinedUpperExtremityStrength",
    "Preop_LeftUpperExtremityWeak", "Index_LeftUpperExtremityStrength", "New_LeftUpperExtremityStrength",
    "Preop_RightUpperExtremityWeak", "Index_RightUpperExtremityStrength", "New_RightUpperExtremityStrength",

    "Preop_CombinedLowerExtremityWeak","Index_CombinedLowerExtremityStrength", "New_CombinedLowerExtremityStrength",
    "Preop_LeftLowerExtremityWeak", "Index_LeftLowerExtremityStrength", "New_LeftLowerExtremityStrength",
    "Preop_RightLowerExtremityWeak", "Index_RightLowerExtremityStrength", "New_RightLowerExtremityStrength",


    "Preop_LighttouchEntireBody", "Index_DeficitToLightTouchEntireBody", "New_LighttouchEntireBody",

    "Preop_CombinedUpperDLTExtremities", "Index_CombinedUpperDLTExtremities", "New_CombinedUpperDLTExtremities",
    "Preop_DLTLeftUpperExtremities", "Index_LightTouchSensoryDeficitLUE", "New_DLTLeftUpperExtremities",
    "Preop_DLTRightUpperExtremities", "Index_LightTouchSensoryDeficitRUE", "New_DLTRightUpperExtremities",

    "Preop_CombinedLowerDLTExtremities", "Index_CombinedLowerDLTExtremities", "New_CombinedLowerDLTExtremities",
    "Preop_DLTLeftLowerExtremities", "Index_LightTouchSensoryDeficitLLE", "New_DLTLeftLowerExtremities",
    "Preop_DLTRightLowerExtremities", "Index_LightTouchSensoryDeficitRLE", "New_DLTRightLowerExtremities",

    "Preop_DeficitToPinprickUpperLeft", "Index_LUEDeficitPinprick", "New_DeficitToPinprickUpperLeft",
    "Preop_DeficitToPinprickUpperRight", "Index_RUEDeficitPinprick", "New_DeficitToPinprickUpperRight",
    "Preop_DeficittoPinprickLowerLeft", "Index_LLEDeficitPinprick", "New_DeficittoPinprickLowerLeft",
    "Preop_DeficittoPinprickLowerRight", "Index_RLEDeficitPinprick", "New_DeficittoPinprickLowerRight",

    "Preop_LightTouchTorso", "Index_DeficitLightTouchTorso", "New_LightTouchTorso",
    "Preop_LightTouchSaddle", "Index_DeficitLIghtTouchSaddle", "New_LightTouchSaddle",


    "Preop_DeepTendonReflexes", "Index_DeepTendonReflexes", "New_DeepTendonReflexes",
   
    "Preop_AnkleClonus", "Index_FUAnkleClonus",
    "Preop_GaitInstability", "Index_GaitStatus",

    "Site",
    "Allocation"
]


In [49]:
DeepTendonReflexes_df = neuro_history[['SubjectId', 'DeepTendonReflexes']]
DeepTendonReflexes_df.columns = ['SubjectId', 'Preop_DeepTendonReflexes_Original']
subject_results_df = subject_results_df.merge(DeepTendonReflexes_df, on='SubjectId', how='left')

In [50]:
subject_results_df.Preop_DeepTendonReflexes_Original.value_counts()

Preop_DeepTendonReflexes_Original
3    113
0     42
1      4
4      3
2      1
Name: count, dtype: int64

In [51]:
encoding_groups_six ={
    "DeepTendonReflexes":("Preop_DeepTendonReflexes_Original", "Index_DeepTendonReflexes"),
}
def custom_index_encoding_six(preop,index):
    if preop == 2 and index == 2:
        return 3
    elif preop == 2 and index in [1,0,3]:
        return 4
    elif preop == 2 and index == 4:
        return 5
    elif preop == 1 and index == 1:
        return 3
    elif preop == 1 and index in [0,3]:
        return 2
    elif preop == 1 and index == 4:
        return 5
    elif preop in [0,3] and index == 1:
        return 4
    elif preop in [0,3] and index in [0,3]:
        return 3
    elif preop in [0,3] and index == 2:
        return 1
    elif preop in [0,3] and index == 4:
        return 5
    elif preop == 4 and index in [0,1,2,3]:
        return 5
# Apply to each mapping
for feature, (preop_col, index_col) in encoding_groups_six.items():
    subject_results_df[f'Index_{feature}'] = subject_results_df.apply(
        lambda row: custom_index_encoding_six(row[preop_col], row[index_col]), axis=1
    )
    subject_results_df[f'Index_{feature}'] = subject_results_df[f'Index_{feature}'].fillna(5)

for feature, (preop_col, index_col) in encoding_groups_six.items():
    print(f"Processing feature: {feature}")
    print(subject_results_df[index_col].value_counts())

Processing feature: DeepTendonReflexes
Index_DeepTendonReflexes
1.0    87
5.0    72
3.0     2
4.0     2
Name: count, dtype: int64


In [52]:
subject_results_df[subject_results_df['SubjectId']==125][['Preop_DeepTendonReflexes','Index_DeepTendonReflexes']]

,Preop_DeepTendonReflexes,Index_DeepTendonReflexes
122,0,3.0


In [53]:
encoding_groups_five ={
    "LeftLowerExtremityStrength":("Preop_LeftLowerExtremityWeak", "Index_LeftLowerExtremityStrength"),
    "RightLowerExtremityStrength":("Preop_RightLowerExtremityWeak", "Index_RightLowerExtremityStrength"),
    "LeftUpperExtremityStrength":("Preop_LeftUpperExtremityWeak", "Index_LeftUpperExtremityStrength"),
    "RightUpperExtremityStrength":("Preop_RightUpperExtremityWeak", "Index_RightUpperExtremityStrength"),
    "DeficitToLightTouchEntireBody":("Preop_LighttouchEntireBody", "Index_DeficitToLightTouchEntireBody"),
    "LUEDeficitPinprick":("Preop_DeficitToPinprickUpperLeft", "Index_LUEDeficitPinprick"),
    "RUEDeficitPinprick":("Preop_DeficitToPinprickUpperRight", "Index_RUEDeficitPinprick"),
    "LLEDeficitPinprick":("Preop_DeficittoPinprickLowerLeft", "Index_LLEDeficitPinprick"),
    "RLEDeficitPinprick":("Preop_DeficittoPinprickLowerRight", "Index_RLEDeficitPinprick"),
    "DeficitLightTouchTorso":("Preop_LightTouchTorso", "Index_DeficitLightTouchTorso"),
    "DeficitLIghtTouchSaddle":("Preop_LightTouchSaddle", "Index_DeficitLIghtTouchSaddle"),
}
def custom_index_encoding_five(preop,index):
    if preop == 0 and index == 1:
        return 3
    elif preop == 0 and index == 0:
        return 2
    elif preop == 0 and index == 2:
        return 3
    elif preop == 0 and index == 3:
        return 4
    elif preop == 0 and index == 4:
        return 5
    elif preop == 1 and index == 1:
        return 1
    elif preop == 1 and index == 0:
        return 2
    elif preop == 1 and index == 2:
        return 3
    elif preop == 1 and index == 3:
        return 4
    elif preop == 1 and index == 4:
        return 5
    elif preop == 1 and index is None:
        return 5
    elif preop == 0 and index is None:
        return 5
# Apply to each mapping
for feature, (preop_col, index_col) in encoding_groups_five.items():
    subject_results_df[f'Index_{feature}'] = subject_results_df.apply(
        lambda row: custom_index_encoding_five(row[preop_col], row[index_col]), axis=1
    )
    subject_results_df[f'Index_{feature}'] = subject_results_df[f'Index_{feature}'].fillna(5)

for feature, (preop_col, index_col) in encoding_groups_five.items():
    print(f"Processing feature: {feature}")
    print(subject_results_df[index_col].value_counts())

Processing feature: LeftLowerExtremityStrength
Index_LeftLowerExtremityStrength
3.0    118
5.0     29
2.0     16
Name: count, dtype: int64
Processing feature: RightLowerExtremityStrength
Index_RightLowerExtremityStrength
3.0    118
5.0     28
2.0     17
Name: count, dtype: int64
Processing feature: LeftUpperExtremityStrength
Index_LeftUpperExtremityStrength
3.0    116
5.0     28
2.0     18
1.0      1
Name: count, dtype: int64
Processing feature: RightUpperExtremityStrength
Index_RightUpperExtremityStrength
3.0    116
5.0     28
2.0     17
1.0      2
Name: count, dtype: int64
Processing feature: DeficitToLightTouchEntireBody
Index_DeficitToLightTouchEntireBody
3.0    106
5.0     29
2.0     28
Name: count, dtype: int64
Processing feature: LUEDeficitPinprick
Index_LUEDeficitPinprick
2.0    125
5.0     28
3.0     10
Name: count, dtype: int64
Processing feature: RUEDeficitPinprick
Index_RUEDeficitPinprick
2.0    125
5.0     28
3.0      9
4.0      1
Name: count, dtype: int64
Processing featu

In [54]:
encoding_groups_four ={
    "FUAnkleClonus":("Preop_AnkleClonus", "Index_FUAnkleClonus"),
}
def custom_index_encoding_four(preop,index):
    if preop == 0 and index == 0:
        return 3
    elif preop == 0 and index == 1:
        return 4
    elif preop == 0 and index == 2:
        return 5
    elif preop == 1 and index == 1:
        return 3
    elif preop == 1 and index == 0:
        return 1
    elif preop == 1 and index == 2:
        return 5
    elif preop == 1 and index is None:
        return 5
    elif preop == 0 and index is None:
        return 5
# Apply to each mapping
for feature, (preop_col, index_col) in encoding_groups_four.items():
    subject_results_df[f'Index_{feature}'] = subject_results_df.apply(
        lambda row: custom_index_encoding_four(row[preop_col], row[index_col]), axis=1
    )
    subject_results_df[f'Index_{feature}'] = subject_results_df[f'Index_{feature}'].fillna(5)

for feature, (preop_col, index_col) in encoding_groups_four.items():
    print(f"Processing feature: {feature}")
    print(subject_results_df[index_col].value_counts())

Processing feature: FUAnkleClonus
Index_FUAnkleClonus
5.0    116
3.0     45
4.0      1
1.0      1
Name: count, dtype: int64


In [55]:
list(subject_results_df)

['SubjectId',
 'Preop_Papilledema',
 'Preop_Nystagmus',
 'Preop_DisconjugateGaze',
 'Preop_ExtraocularPalsies',
 'Preop_FacialWeakness',
 'Preop_DisturbanceInFacialSensation',
 'Preop_HearingLoss',
 'Preop_Hoarseness',
 'Preop_TongueDeviation',
 'Preop_WeakShrug',
 'Preop_LeftUpperExtremityWeak',
 'Preop_RightUpperExtremityWeak',
 'Preop_LeftLowerExtremityWeak',
 'Preop_RightLowerExtremityWeak',
 'Preop_LighttouchEntireBody',
 'Preop_DLTLeftUpperExtremities',
 'Preop_DLTRightUpperExtremities',
 'Preop_DLTLeftLowerExtremities',
 'Preop_DLTRightLowerExtremities',
 'Preop_DeficitToPinprickUpperLeft',
 'Preop_DeficitToPinprickUpperRight',
 'Preop_DeficittoPinprickLowerLeft',
 'Preop_DeficittoPinprickLowerRight',
 'Preop_LightTouchTorso',
 'Preop_LightTouchSaddle',
 'Preop_DeepTendonReflexes',
 'Preop_AnkleClonus',
 'Preop_GaitInstability',
 'Index_Papilledema',
 'Index_Nystagmus',
 'Index_DisconjugateGaze',
 'Index_DocExtraOcPalsies',
 'Index_FacialWeakness',
 'Index_DisturbanceFacSen',
 '

In [56]:
encoding_groups_three ={
    "TongueDeviation":("Preop_TongueDeviation", "Index_TongueDeviation"),
    "WeakShrug":("Preop_WeakShrug", "Index_WeakShrug"),
    "LightTouchSensoryDeficitLLE":("Preop_DLTLeftLowerExtremities","Index_LightTouchSensoryDeficitLLE"),
    "LightTouchSensoryDeficitLUE":("Preop_DLTLeftUpperExtremities","Index_LightTouchSensoryDeficitLUE"),
    "LightTouchSensoryDeficitRLE":('Preop_DLTRightLowerExtremities','Index_LightTouchSensoryDeficitRLE'),
    "LightTouchSensoryDeficitRUE":('Preop_DLTRightUpperExtremities','Index_LightTouchSensoryDeficitRUE')
}
def custom_index_encoding_three(preop,index):
    if preop == 0 and index == 0:
        return 2
    elif preop == 0 and index == 1:
        return 3
    elif preop == 0 and index == 2:
        return 3
    elif preop == 0 and index == 3:
        return 4
    elif preop == 0 and index == 4:
        return 5
    elif preop == 0 and index is None:
        return 5
    elif preop == 1 and index == 0:
        return 2
    elif preop == 1 and index == 1:
        return 1
    elif preop == 1 and index == 2:
        return 3
    elif preop == 1 and index == 3:
        return 4
    elif preop == 1 and index == 4:
        return 5
    elif preop == 1 and index is None:
        return 5
# Apply to each mapping
for feature, (preop_col, index_col) in encoding_groups_three.items():
    subject_results_df[f'Index_{feature}'] = subject_results_df.apply(
        lambda row: custom_index_encoding_three(row[preop_col], row[index_col]), axis=1
    )
    subject_results_df[f'Index_{feature}'] = subject_results_df[f'Index_{feature}'].fillna(5)

for feature, (preop_col, index_col) in encoding_groups_three.items():
    print(f"Processing feature: {feature}")
    print(subject_results_df[index_col].value_counts())

Processing feature: TongueDeviation
Index_TongueDeviation
3.0    112
5.0     51
Name: count, dtype: int64
Processing feature: WeakShrug
Index_WeakShrug
3.0    95
5.0    68
Name: count, dtype: int64
Processing feature: LightTouchSensoryDeficitLLE
Index_LightTouchSensoryDeficitLLE
3.0    98
2.0    37
5.0    28
Name: count, dtype: int64
Processing feature: LightTouchSensoryDeficitLUE
Index_LightTouchSensoryDeficitLUE
3.0    97
2.0    37
5.0    28
1.0     1
Name: count, dtype: int64
Processing feature: LightTouchSensoryDeficitRLE
Index_LightTouchSensoryDeficitRLE
3.0    98
2.0    37
5.0    28
Name: count, dtype: int64
Processing feature: LightTouchSensoryDeficitRUE
Index_LightTouchSensoryDeficitRUE
3.0    94
2.0    38
5.0    28
1.0     2
4.0     1
Name: count, dtype: int64


In [57]:
# Explicit mapping between Preop and Index features
encoding_groups_two = {
    "HearingLoss": ("Preop_HearingLoss", "Index_HearingLoss"),
    "GaitStatus": ("Preop_GaitInstability", "Index_GaitStatus"),
}

def custom_index_encoding_two(preop, index):
    if preop == 0 and index == 0:
        return 1
    elif preop ==0 and index == 1:
        return 3 
    elif preop == 0 and index == 2:
        return 3
    elif preop == 0 and index == 3:
        return 4
    elif preop == 0 and index == 4:
        return 5
    elif preop == 0 and index == None:
        return 5
    elif preop == 1 and index == 0:
        return 1
    elif preop == 1 and index == 1:
        return 1
    elif preop == 1 and index == 2:
        return 3
    elif preop == 1 and index == 3:
        return 4
    elif preop == 1 and index == 4:
        return 5
    elif preop == 1 and index == None:
        return 5

# Apply to each mapping
for feature, (preop_col, index_col) in encoding_groups_two.items():
    subject_results_df[f'Index_{feature}'] = subject_results_df.apply(
        lambda row: custom_index_encoding_two(row[preop_col], row[index_col]), axis=1
    )
    subject_results_df[f'Index_{feature}'] = subject_results_df[f'Index_{feature}'].fillna(5)

for feature, (preop_col, index_col) in encoding_groups_two.items():
    print(f"Processing feature: {feature}")
    print(subject_results_df[index_col].value_counts())

Processing feature: HearingLoss
Index_HearingLoss
3.0    94
1.0    41
5.0    28
Name: count, dtype: int64
Processing feature: GaitStatus
Index_GaitStatus
3.0    106
5.0     49
1.0      5
4.0      3
Name: count, dtype: int64


In [58]:
subject_results_df

,SubjectId,Preop_Papilledema,Preop_Nystagmus,Preop_DisconjugateGaze,Preop_ExtraocularPalsies,Preop_FacialWeakness,Preop_DisturbanceInFacialSensation,Preop_HearingLoss,Preop_Hoarseness,Preop_TongueDeviation,...,Index_CombinedUpperExtremityStrength,Index_CombinedLowerExtremityStrength,Index_CombinedUpperDLTExtremities,Index_CombinedLowerDLTExtremities,New_CombinedUpperExtremityStrength,New_CombinedLowerExtremityStrength,New_CombinedUpperDLTExtremities,New_CombinedLowerDLTExtremities,Allocation,Preop_DeepTendonReflexes_Original
0,1,0,0,0,0,0,0,0,0,0,...,5,5,5,5,5,5,5,5,PFD,0
1,2,0,0,0,0,0,0,0,0,0,...,1,1,1,1,0,0,0,0,PFD,3
2,3,0,0,0,0,0,0,0,0,0,...,1,1,1,1,0,0,0,0,PFDD,0
3,4,0,0,0,0,0,0,0,0,0,...,1,1,1,1,0,0,0,0,PFDD,0
4,5,0,0,0,0,0,0,0,0,0,...,5,5,5,5,5,5,5,5,PFD,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158,164,0,0,0,0,0,0,0,0,0,...,5,5,5,5,5,5,5,5,PFD,3
159,165,0,0,0,0,0,0,0,0,0,...,1,1,1,1,0,0,0,0,PFDD,3
160,166,0,0,0,0,0,0,0,0,0,...,5,5,5,5,5,5,5,5,PFD,3
161,167,0,0,0,0,0,0,0,0,0,...,5,5,5,5,5,5,5,5,PFD,0


In [59]:
subject_results_df = subject_results_df.dropna(subset=['Allocation'])

In [60]:
missing_summary = subject_results_df.isnull().sum()
print("Missing values per column in subject_results_df:")
print(missing_summary[missing_summary > 0])

Missing values per column in subject_results_df:
Series([], dtype: int64)


In [61]:
list(subject_results_df)

['SubjectId',
 'Preop_Papilledema',
 'Preop_Nystagmus',
 'Preop_DisconjugateGaze',
 'Preop_ExtraocularPalsies',
 'Preop_FacialWeakness',
 'Preop_DisturbanceInFacialSensation',
 'Preop_HearingLoss',
 'Preop_Hoarseness',
 'Preop_TongueDeviation',
 'Preop_WeakShrug',
 'Preop_LeftUpperExtremityWeak',
 'Preop_RightUpperExtremityWeak',
 'Preop_LeftLowerExtremityWeak',
 'Preop_RightLowerExtremityWeak',
 'Preop_LighttouchEntireBody',
 'Preop_DLTLeftUpperExtremities',
 'Preop_DLTRightUpperExtremities',
 'Preop_DLTLeftLowerExtremities',
 'Preop_DLTRightLowerExtremities',
 'Preop_DeficitToPinprickUpperLeft',
 'Preop_DeficitToPinprickUpperRight',
 'Preop_DeficittoPinprickLowerLeft',
 'Preop_DeficittoPinprickLowerRight',
 'Preop_LightTouchTorso',
 'Preop_LightTouchSaddle',
 'Preop_DeepTendonReflexes',
 'Preop_AnkleClonus',
 'Preop_GaitInstability',
 'Index_Papilledema',
 'Index_Nystagmus',
 'Index_DisconjugateGaze',
 'Index_DocExtraOcPalsies',
 'Index_FacialWeakness',
 'Index_DisturbanceFacSen',
 '

In [62]:
# Explicit mapping between Preop and Index features
encoding_groups = {
    "DisturbanceFacSen": ("Preop_DisturbanceInFacialSensation", "Index_DisturbanceFacSen"),
    "Nystagmus": ("Preop_Nystagmus", "Index_Nystagmus"),
    "DisconjugateGaze": ("Preop_DisconjugateGaze", "Index_DisconjugateGaze"),
    "DocExtraOcPalsies": ("Preop_ExtraocularPalsies", "Index_DocExtraOcPalsies"),
    "FacialWeakness": ("Preop_FacialWeakness", "Index_FacialWeakness"),
    "Hoarseness": ("Preop_Hoarseness", "Index_Hoarseness")
}

def custom_index_encoding(preop, index):
    if preop == 0 and index == 1:
        return 3  
    elif preop == 0 and index == 2:
        return 2  
    elif preop == 0 and index == 3:
        return 3
    elif preop == 0 and index == 4:
        return 4
    elif preop == 0 and index == 0:
        return 5
    elif preop == 0 and index is None:
        return 3
    elif preop == 1 and index == 1:
        return 1  
    elif preop == 1 and index == 2:
        return 2  
    elif preop == 1 and index == 3:
        return 3  
    elif preop == 1 and index == 4:
        return 4  
    elif preop == 1 and index == 0:
        return 5
    elif preop == 1 and index is None:
        return 5
    return None  # fallback

# Apply to each mapping
for feature, (preop_col, index_col) in encoding_groups.items():
    subject_results_df[f'Index_{feature}'] = subject_results_df.apply(
        lambda row: custom_index_encoding(row[preop_col], row[index_col]), axis=1
    )
    subject_results_df[f'Index_{feature}'] = subject_results_df[f'Index_{feature}'].fillna(5)


/var/folders/5b/z7m8dzcs5td_8q7jrcf_9spm0000gn/T/ipykernel_20362/2251933073.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subject_results_df[f'Index_{feature}'] = subject_results_df.apply(
/var/folders/5b/z7m8dzcs5td_8q7jrcf_9spm0000gn/T/ipykernel_20362/2251933073.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subject_results_df[f'Index_{feature}'] = subject_results_df[f'Index_{feature}'].fillna(5)


In [63]:
for feature, (preop_col, index_col) in encoding_groups.items():
    print(f"Processing feature: {feature}")
    print(subject_results_df[index_col].value_counts())

Processing feature: DisturbanceFacSen
Index_DisturbanceFacSen
3.0    84
5.0    77
1.0     1
Name: count, dtype: int64
Processing feature: Nystagmus
Index_Nystagmus
3.0    111
5.0     47
1.0      3
4.0      1
Name: count, dtype: int64
Processing feature: DisconjugateGaze
Index_DisconjugateGaze
3.0    124
5.0     36
1.0      1
4.0      1
Name: count, dtype: int64
Processing feature: DocExtraOcPalsies
Index_DocExtraOcPalsies
3.0    123
5.0     36
1.0      3
Name: count, dtype: int64
Processing feature: FacialWeakness
Index_FacialWeakness
3.0    125
5.0     37
Name: count, dtype: int64
Processing feature: Hoarseness
Index_Hoarseness
3.0    88
5.0    73
1.0     1
Name: count, dtype: int64


In [64]:
list(subject_results_df)

['SubjectId',
 'Preop_Papilledema',
 'Preop_Nystagmus',
 'Preop_DisconjugateGaze',
 'Preop_ExtraocularPalsies',
 'Preop_FacialWeakness',
 'Preop_DisturbanceInFacialSensation',
 'Preop_HearingLoss',
 'Preop_Hoarseness',
 'Preop_TongueDeviation',
 'Preop_WeakShrug',
 'Preop_LeftUpperExtremityWeak',
 'Preop_RightUpperExtremityWeak',
 'Preop_LeftLowerExtremityWeak',
 'Preop_RightLowerExtremityWeak',
 'Preop_LighttouchEntireBody',
 'Preop_DLTLeftUpperExtremities',
 'Preop_DLTRightUpperExtremities',
 'Preop_DLTLeftLowerExtremities',
 'Preop_DLTRightLowerExtremities',
 'Preop_DeficitToPinprickUpperLeft',
 'Preop_DeficitToPinprickUpperRight',
 'Preop_DeficittoPinprickLowerLeft',
 'Preop_DeficittoPinprickLowerRight',
 'Preop_LightTouchTorso',
 'Preop_LightTouchSaddle',
 'Preop_DeepTendonReflexes',
 'Preop_AnkleClonus',
 'Preop_GaitInstability',
 'Index_Papilledema',
 'Index_Nystagmus',
 'Index_DisconjugateGaze',
 'Index_DocExtraOcPalsies',
 'Index_FacialWeakness',
 'Index_DisturbanceFacSen',
 '

In [65]:
list(subject_results_df)

['SubjectId',
 'Preop_Papilledema',
 'Preop_Nystagmus',
 'Preop_DisconjugateGaze',
 'Preop_ExtraocularPalsies',
 'Preop_FacialWeakness',
 'Preop_DisturbanceInFacialSensation',
 'Preop_HearingLoss',
 'Preop_Hoarseness',
 'Preop_TongueDeviation',
 'Preop_WeakShrug',
 'Preop_LeftUpperExtremityWeak',
 'Preop_RightUpperExtremityWeak',
 'Preop_LeftLowerExtremityWeak',
 'Preop_RightLowerExtremityWeak',
 'Preop_LighttouchEntireBody',
 'Preop_DLTLeftUpperExtremities',
 'Preop_DLTRightUpperExtremities',
 'Preop_DLTLeftLowerExtremities',
 'Preop_DLTRightLowerExtremities',
 'Preop_DeficitToPinprickUpperLeft',
 'Preop_DeficitToPinprickUpperRight',
 'Preop_DeficittoPinprickLowerLeft',
 'Preop_DeficittoPinprickLowerRight',
 'Preop_LightTouchTorso',
 'Preop_LightTouchSaddle',
 'Preop_DeepTendonReflexes',
 'Preop_AnkleClonus',
 'Preop_GaitInstability',
 'Index_Papilledema',
 'Index_Nystagmus',
 'Index_DisconjugateGaze',
 'Index_DocExtraOcPalsies',
 'Index_FacialWeakness',
 'Index_DisturbanceFacSen',
 '

In [66]:
def add_refined_from_ordered(df, ordered_cols):
    """
    For each symptom in ordered_cols:
      - If New_ col exists and == 1 → Refined_ = 4
      - Else Refined_ = Index_
    """
    for col in ordered_cols:
        if col.startswith("Index_"):
            base = col.replace("Index_", "")
            refined_col = f"Refined_{base}"

            # General case
            new_col = f"New_{base}"
            if new_col in df.columns:
                df[refined_col] = df.apply(
                    lambda row: 4 if row[new_col] == 1 else row[col],
                    axis=1
                )
            else:
                df[refined_col] = df[col]
        

    return df



subject_results_df = add_refined_from_ordered(subject_results_df, ordered_cols)


/var/folders/5b/z7m8dzcs5td_8q7jrcf_9spm0000gn/T/ipykernel_20362/2194610883.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[refined_col] = df.apply(
/var/folders/5b/z7m8dzcs5td_8q7jrcf_9spm0000gn/T/ipykernel_20362/2194610883.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[refined_col] = df.apply(
/var/folders/5b/z7m8dzcs5td_8q7jrcf_9spm0000gn/T/ipykernel_20362/2194610883.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_inde

In [67]:
def custom_index_papilledema(row):
    preop = row['Preop_Papilledema']
    index = row['Index_Papilledema']
    # preop is absent (0), index is absent (1) -> stable (3)
    if preop == 0 and index == 1:
        return 3
    elif preop == 0 and index == 0:
        return 4
    elif preop == 1 and index == 1:
        return 1
    elif preop == 1 and index == 0:
        return 4
    elif preop == 1 and index == 2:
        return 5
    elif preop == 0 and index == 2:
        return 5
    elif preop == 0 and index is None:
        return 5
    elif preop == 1 and index is None:
        return 5
    
def custom_refined_papilledema(row):
    preop = row['Preop_Papilledema']
    index = row['Index_Papilledema']
    # Case 1: both are 1 → 3 preop/index both present
    if preop == 1 and index == 1:
        return 3
    # Case 2: both are 0 → 3 preop/index both absent
    elif preop == 0 and index == 0:
        return 3
    # Case 3: preop 0, index 1 → 4
    elif preop == 0 and index == 1:
        return 4
    elif preop == 1 and index == 0:
        return 1
    elif preop == 0 and index == 3:
        return 3
    elif preop == 1 and index == 3:
        return 3
    elif preop == 1 and index == 5:
        return 5
    elif preop == 0 and index == 5:
        return 3
subject_results_df['Index_Papilledema'] = subject_results_df.apply(custom_index_papilledema, axis=1)
subject_results_df.Index_Papilledema.value_counts()
subject_results_df['Index_Papilledema'] = subject_results_df['Index_Papilledema'].fillna(3)
subject_results_df.Index_Papilledema.value_counts()
subject_results_df['Refined_Papilledema'] = subject_results_df.apply(custom_refined_papilledema, axis=1)




/var/folders/5b/z7m8dzcs5td_8q7jrcf_9spm0000gn/T/ipykernel_20362/1672421416.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subject_results_df['Index_Papilledema'] = subject_results_df.apply(custom_index_papilledema, axis=1)
/var/folders/5b/z7m8dzcs5td_8q7jrcf_9spm0000gn/T/ipykernel_20362/1672421416.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subject_results_df['Index_Papilledema'] = subject_results_df['Index_Papilledema'].fillna(3)
/var/folders/5b/z7m8dzcs5td_8q7jrcf_9spm0000gn/T/ipykernel_20

In [68]:
subject_results_df.Index_Papilledema.value_counts()

Index_Papilledema
5.0    88
3.0    74
Name: count, dtype: int64

In [69]:
subject_results_df.Refined_Papilledema.value_counts()

Refined_Papilledema
3    161
5      1
Name: count, dtype: int64

In [70]:
def enforce_stable_rule_from_order(df: pd.DataFrame, ordered_cols: list[str]) -> pd.DataFrame:
    """
    For each Index_* in ordered_cols (except Papilledema):
      if (nearest preceding Preop_* == 0) and (Index_* == 1) and (New_* == 0 or missing) -> Refined_* = 3.
    If Refined_* doesn't exist yet, initialize it from Index_* before applying the rule.
    """
    # Precompute positions for quick backtracking to the nearest Preop_ before each Index_
    pos = {c: i for i, c in enumerate(ordered_cols)}

    for idx_col in [c for c in ordered_cols if c.startswith("Index_")]:
        base = idx_col.replace("Index_", "")
        if base == "Papilledema":
            continue  # skip Papilledema per your requirement

        # Find the nearest preceding Preop_* in ordered_cols
        preop_col = None
        i = pos[idx_col] - 1
        while i >= 0:
            c = ordered_cols[i]
            if c.startswith("Preop_"):
                preop_col = c
                break
            i -= 1

        # If we don't find a preop col or columns are missing in df, skip
        if (preop_col is None) or (preop_col not in df.columns) or (idx_col not in df.columns):
            continue

        new_col = f"New_{base}"
        refined_col = f"Refined_{base}"

        # # If Refined_* not present, initialize from Index_ first
        # if refined_col not in df.columns:
        #     df[refined_col] = df[idx_col]
        #     df[refined_col] = df[refined_col].replace({0: 5})
        


        # Masks
        preop_zero_mask = (df[preop_col] == 0)
        index_one_mask  = (df[idx_col] == 5)
        if new_col in df.columns:
            new_zero_mask = (df[new_col] != 1)  # treat non-1 (including NaN) as 0 for this rule
        else:
            new_zero_mask = pd.Series(True, index=df.index)

        # Apply rule → set stable (3)
        mask = preop_zero_mask & index_one_mask & new_zero_mask
        df.loc[mask, refined_col] = 3

        preop_zero_mask = (df[preop_col] == 0)
        index_one_mask  = (df[idx_col] == 1) 
        if new_col in df.columns:
            new_zero_mask = (df[new_col] != 1)  
        else:
            new_zero_mask = pd.Series(True, index=df.index)

        mask = preop_zero_mask & index_one_mask & new_zero_mask
        df.loc[mask, refined_col] = 3

        preop_zero_mask = (df[preop_col] == 0)
        index_one_mask  = (df[idx_col] == 2) 
        if new_col in df.columns:
            new_zero_mask = (df[new_col] != 1)  
        else:
            new_zero_mask = pd.Series(True, index=df.index)

        mask = preop_zero_mask & index_one_mask & new_zero_mask
        df.loc[mask, refined_col] = 3

    return df


subject_results_df = enforce_stable_rule_from_order(subject_results_df, ordered_cols)

In [71]:
list(subject_results_df)

['SubjectId',
 'Preop_Papilledema',
 'Preop_Nystagmus',
 'Preop_DisconjugateGaze',
 'Preop_ExtraocularPalsies',
 'Preop_FacialWeakness',
 'Preop_DisturbanceInFacialSensation',
 'Preop_HearingLoss',
 'Preop_Hoarseness',
 'Preop_TongueDeviation',
 'Preop_WeakShrug',
 'Preop_LeftUpperExtremityWeak',
 'Preop_RightUpperExtremityWeak',
 'Preop_LeftLowerExtremityWeak',
 'Preop_RightLowerExtremityWeak',
 'Preop_LighttouchEntireBody',
 'Preop_DLTLeftUpperExtremities',
 'Preop_DLTRightUpperExtremities',
 'Preop_DLTLeftLowerExtremities',
 'Preop_DLTRightLowerExtremities',
 'Preop_DeficitToPinprickUpperLeft',
 'Preop_DeficitToPinprickUpperRight',
 'Preop_DeficittoPinprickLowerLeft',
 'Preop_DeficittoPinprickLowerRight',
 'Preop_LightTouchTorso',
 'Preop_LightTouchSaddle',
 'Preop_DeepTendonReflexes',
 'Preop_AnkleClonus',
 'Preop_GaitInstability',
 'Index_Papilledema',
 'Index_Nystagmus',
 'Index_DisconjugateGaze',
 'Index_DocExtraOcPalsies',
 'Index_FacialWeakness',
 'Index_DisturbanceFacSen',
 '

In [72]:
print(subject_results_df['Refined_LeftLowerExtremityStrength'].value_counts())
print(subject_results_df['Refined_RightLowerExtremityStrength'].value_counts())
print(subject_results_df['Refined_LeftUpperExtremityStrength'].value_counts())
print(subject_results_df['Refined_RightUpperExtremityStrength'].value_counts())

Refined_LeftLowerExtremityStrength
3.0    161
4.0      1
Name: count, dtype: int64
Refined_RightLowerExtremityStrength
3.0    162
Name: count, dtype: int64
Refined_LeftUpperExtremityStrength
3.0    161
1.0      1
Name: count, dtype: int64
Refined_RightUpperExtremityStrength
3.0    159
1.0      2
2.0      1
Name: count, dtype: int64


In [73]:
print(subject_results_df['Refined_DocExtraOcPalsies'].value_counts())

Refined_DocExtraOcPalsies
3.0    159
1.0      3
Name: count, dtype: int64


In [74]:
print(subject_results_df['Preop_TongueDeviation'].value_counts())
print(subject_results_df['Refined_TongueDeviation'].value_counts())

Preop_TongueDeviation
0    162
Name: count, dtype: int64
Refined_TongueDeviation
3.0    162
Name: count, dtype: int64


In [75]:
print(subject_results_df['Preop_WeakShrug'].value_counts())
print(subject_results_df['Refined_WeakShrug'].value_counts())

Preop_WeakShrug
0    162
Name: count, dtype: int64
Refined_WeakShrug
3.0    162
Name: count, dtype: int64


In [76]:
subject_results_df['Refined_WeakShrug'].value_counts()

Refined_WeakShrug
3.0    162
Name: count, dtype: int64

In [77]:
subject_results_df['Index_HearingLoss'].value_counts()

Index_HearingLoss
3.0    94
1.0    41
5.0    27
Name: count, dtype: int64

In [78]:
subject_results_df['Refined_HearingLoss'].value_counts()

Refined_HearingLoss
3.0    160
5.0      1
1.0      1
Name: count, dtype: int64

In [79]:
subject_results_df['Refined_GaitStatus'].value_counts()

Refined_GaitStatus
3.0    148
5.0      6
1.0      5
4.0      3
Name: count, dtype: int64

In [80]:
subject_results_df['Index_Papilledema'].value_counts()

Index_Papilledema
5.0    88
3.0    74
Name: count, dtype: int64

In [81]:
subject_results_df['Preop_Papilledema'].value_counts()

Preop_Papilledema
0    160
1      2
Name: count, dtype: int64

In [82]:
subject_results_df['Preop_Papilledema'].value_counts()
subject_results_df['Index_Papilledema'].value_counts()

Index_Papilledema
5.0    88
3.0    74
Name: count, dtype: int64

In [83]:
subject_results_df['Index_Papilledema'].value_counts()

Index_Papilledema
5.0    88
3.0    74
Name: count, dtype: int64

In [84]:
subject_results_df['Refined_Papilledema'].value_counts()

Refined_Papilledema
3    161
5      1
Name: count, dtype: int64

In [85]:
subject_results_df

,SubjectId,Preop_Papilledema,Preop_Nystagmus,Preop_DisconjugateGaze,Preop_ExtraocularPalsies,Preop_FacialWeakness,Preop_DisturbanceInFacialSensation,Preop_HearingLoss,Preop_Hoarseness,Preop_TongueDeviation,...,Refined_LightTouchSensoryDeficitRLE,Refined_LUEDeficitPinprick,Refined_RUEDeficitPinprick,Refined_LLEDeficitPinprick,Refined_RLEDeficitPinprick,Refined_DeficitLightTouchTorso,Refined_DeficitLIghtTouchSaddle,Refined_DeepTendonReflexes,Refined_FUAnkleClonus,Refined_GaitStatus
0,1,0,0,0,0,0,0,0,0,0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0
1,2,0,0,0,0,0,0,0,0,0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0
2,3,0,0,0,0,0,0,0,0,0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0
3,4,0,0,0,0,0,0,0,0,0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0
4,5,0,0,0,0,0,0,0,0,0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158,164,0,0,0,0,0,0,0,0,0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0
159,165,0,0,0,0,0,0,0,0,0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0
160,166,0,0,0,0,0,0,0,0,0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0
161,167,0,0,0,0,0,0,0,0,0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0


In [86]:
subject_results_df['Refined_FUAnkleClonus'].value_counts()

Refined_FUAnkleClonus
3.0    158
5.0      2
4.0      1
1.0      1
Name: count, dtype: int64

In [87]:
missing_counts = subject_results_df.isnull().sum()
print(missing_counts[missing_counts > 0])

Series([], dtype: int64)


In [88]:
list(subject_results_df)

['SubjectId',
 'Preop_Papilledema',
 'Preop_Nystagmus',
 'Preop_DisconjugateGaze',
 'Preop_ExtraocularPalsies',
 'Preop_FacialWeakness',
 'Preop_DisturbanceInFacialSensation',
 'Preop_HearingLoss',
 'Preop_Hoarseness',
 'Preop_TongueDeviation',
 'Preop_WeakShrug',
 'Preop_LeftUpperExtremityWeak',
 'Preop_RightUpperExtremityWeak',
 'Preop_LeftLowerExtremityWeak',
 'Preop_RightLowerExtremityWeak',
 'Preop_LighttouchEntireBody',
 'Preop_DLTLeftUpperExtremities',
 'Preop_DLTRightUpperExtremities',
 'Preop_DLTLeftLowerExtremities',
 'Preop_DLTRightLowerExtremities',
 'Preop_DeficitToPinprickUpperLeft',
 'Preop_DeficitToPinprickUpperRight',
 'Preop_DeficittoPinprickLowerLeft',
 'Preop_DeficittoPinprickLowerRight',
 'Preop_LightTouchTorso',
 'Preop_LightTouchSaddle',
 'Preop_DeepTendonReflexes',
 'Preop_AnkleClonus',
 'Preop_GaitInstability',
 'Index_Papilledema',
 'Index_Nystagmus',
 'Index_DisconjugateGaze',
 'Index_DocExtraOcPalsies',
 'Index_FacialWeakness',
 'Index_DisturbanceFacSen',
 '

In [89]:
ordered_cols_refined = [
    "SubjectId",

    "Preop_Papilledema", "Index_Papilledema", "New_Papilledema", "Refined_Papilledema",
    "Preop_Nystagmus", "Index_Nystagmus", "New_Nystagmus", "Refined_Nystagmus",
    "Preop_DisconjugateGaze", "Index_DisconjugateGaze", "New_DisconjugateGaze", "Refined_DisconjugateGaze",
    "Preop_ExtraocularPalsies", "Index_DocExtraOcPalsies", "New_ExtraocularPalsies", "Refined_DocExtraOcPalsies",
    "Preop_FacialWeakness", "Index_FacialWeakness", "New_FacialWeakness", "Refined_FacialWeakness",
    "Preop_DisturbanceInFacialSensation", "Index_DisturbanceFacSen", "New_DisturbanceInFacialSensation", "Refined_DisturbanceFacSen",
    "Preop_HearingLoss", "Index_HearingLoss", "New_HearingLoss", "Refined_HearingLoss",
    "Preop_Hoarseness", "Index_Hoarseness", "New_Hoarseness", "Refined_Hoarseness",
    "Preop_TongueDeviation", "Index_TongueDeviation", "New_TongueDeviation", "Refined_TongueDeviation",
    "Preop_WeakShrug", "Index_WeakShrug", "New_WeakShrug", "Refined_WeakShrug",

    "Preop_CombinedUpperExtremityWeak", "Index_CombinedUpperExtremityStrength", "New_CombinedUpperExtremityStrength", "Refined_CombinedUpperExtremityStrength",
    "Preop_LeftUpperExtremityWeak", "Index_LeftUpperExtremityStrength", "New_LeftUpperExtremityStrength", "Refined_LeftUpperExtremityStrength",
    "Preop_RightUpperExtremityWeak", "Index_RightUpperExtremityStrength", "New_RightUpperExtremityStrength", "Refined_RightUpperExtremityStrength",

    "Preop_CombinedLowerExtremityWeak", "Index_CombinedLowerExtremityStrength", "New_CombinedLowerExtremityStrength", "Refined_CombinedLowerExtremityStrength",
    "Preop_LeftLowerExtremityWeak", "Index_LeftLowerExtremityStrength", "New_LeftLowerExtremityStrength", "Refined_LeftLowerExtremityStrength",
    "Preop_RightLowerExtremityWeak", "Index_RightLowerExtremityStrength", "New_RightLowerExtremityStrength", "Refined_RightLowerExtremityStrength",

    "Preop_LighttouchEntireBody", "Index_DeficitToLightTouchEntireBody", "New_LighttouchEntireBody", "Refined_DeficitToLightTouchEntireBody",

    "Preop_CombinedUpperDLTExtremities", "Index_CombinedUpperDLTExtremities", "New_CombinedUpperDLTExtremities", "Refined_CombinedUpperDLTExtremities",
    "Preop_DLTLeftUpperExtremities", "Index_LightTouchSensoryDeficitLUE", "New_DLTLeftUpperExtremities", "Refined_LightTouchSensoryDeficitLUE",
    "Preop_DLTRightUpperExtremities", "Index_LightTouchSensoryDeficitRUE", "New_DLTRightUpperExtremities", "Refined_LightTouchSensoryDeficitRUE",

    "Preop_CombinedLowerDLTExtremities", "Index_CombinedLowerDLTExtremities", "New_CombinedLowerDLTExtremities", "Refined_CombinedLowerDLTExtremities",
    "Preop_DLTLeftLowerExtremities", "Index_LightTouchSensoryDeficitLLE", "New_DLTLeftLowerExtremities", "Refined_LightTouchSensoryDeficitLLE",
    "Preop_DLTRightLowerExtremities", "Index_LightTouchSensoryDeficitRLE", "New_DLTRightLowerExtremities", "Refined_LightTouchSensoryDeficitRLE",

    "Preop_DeficitToPinprickUpperLeft", "Index_LUEDeficitPinprick", "New_DeficitToPinprickUpperLeft", "Refined_LUEDeficitPinprick",
    "Preop_DeficitToPinprickUpperRight", "Index_RUEDeficitPinprick", "New_DeficitToPinprickUpperRight", "Refined_RUEDeficitPinprick",
    "Preop_DeficittoPinprickLowerLeft", "Index_LLEDeficitPinprick", "New_DeficittoPinprickLowerLeft", "Refined_LLEDeficitPinprick",
    "Preop_DeficittoPinprickLowerRight", "Index_RLEDeficitPinprick", "New_DeficittoPinprickLowerRight", "Refined_RLEDeficitPinprick",

    "Preop_LightTouchTorso", "Index_DeficitLightTouchTorso", "New_LightTouchTorso", "Refined_DeficitLightTouchTorso",
    "Preop_LightTouchSaddle", "Index_DeficitLIghtTouchSaddle", "New_LightTouchSaddle", "Refined_DeficitLIghtTouchSaddle",

    "Preop_DeepTendonReflexes", "Index_DeepTendonReflexes", "New_DeepTendonReflexes", "Refined_DeepTendonReflexes",
    "Preop_AnkleClonus", "Index_FUAnkleClonus", "New_AnkleClonus", "Refined_FUAnkleClonus",
    "Preop_GaitInstability", "Index_GaitStatus", "New_GaitInstability", "Refined_GaitStatus",

    "Site",
    "Allocation",
]
# Reorder columns
subject_results_df = subject_results_df[[col for col in ordered_cols_refined if col in subject_results_df.columns]]
subject_results_df

,SubjectId,Preop_Papilledema,Index_Papilledema,New_Papilledema,Refined_Papilledema,Preop_Nystagmus,Index_Nystagmus,New_Nystagmus,Refined_Nystagmus,Preop_DisconjugateGaze,...,Preop_AnkleClonus,Index_FUAnkleClonus,New_AnkleClonus,Refined_FUAnkleClonus,Preop_GaitInstability,Index_GaitStatus,New_GaitInstability,Refined_GaitStatus,Site,Allocation
0,1,0,3.0,5.0,3,0,5.0,5.0,3.0,0,...,0,5.0,5.0,3.0,0,5.0,5.0,3.0,1.0,PFD
1,2,0,3.0,5.0,3,0,3.0,5.0,3.0,0,...,0,3.0,5.0,3.0,0,3.0,5.0,3.0,1.0,PFD
2,3,0,5.0,5.0,3,0,3.0,5.0,3.0,0,...,0,5.0,5.0,3.0,0,5.0,5.0,3.0,33.0,PFDD
3,4,0,5.0,5.0,3,0,3.0,5.0,3.0,0,...,0,5.0,5.0,3.0,0,3.0,5.0,3.0,33.0,PFDD
4,5,0,5.0,5.0,3,0,3.0,5.0,3.0,0,...,0,5.0,5.0,3.0,0,5.0,5.0,3.0,1.0,PFD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158,164,0,3.0,5.0,3,0,5.0,5.0,3.0,0,...,0,5.0,5.0,3.0,0,5.0,5.0,3.0,18.0,PFD
159,165,0,5.0,5.0,3,0,3.0,5.0,3.0,0,...,0,5.0,5.0,3.0,0,3.0,5.0,3.0,28.0,PFDD
160,166,0,3.0,5.0,3,0,5.0,5.0,3.0,0,...,0,5.0,5.0,3.0,0,5.0,5.0,3.0,7.0,PFD
161,167,0,5.0,5.0,3,0,5.0,5.0,3.0,0,...,0,5.0,5.0,3.0,0,5.0,5.0,3.0,7.0,PFD


In [90]:
subject_results_df['Refined_FUAnkleClonus'].value_counts()

Refined_FUAnkleClonus
3.0    158
5.0      2
4.0      1
1.0      1
Name: count, dtype: int64

In [91]:
subject_results_df

,SubjectId,Preop_Papilledema,Index_Papilledema,New_Papilledema,Refined_Papilledema,Preop_Nystagmus,Index_Nystagmus,New_Nystagmus,Refined_Nystagmus,Preop_DisconjugateGaze,...,Preop_AnkleClonus,Index_FUAnkleClonus,New_AnkleClonus,Refined_FUAnkleClonus,Preop_GaitInstability,Index_GaitStatus,New_GaitInstability,Refined_GaitStatus,Site,Allocation
0,1,0,3.0,5.0,3,0,5.0,5.0,3.0,0,...,0,5.0,5.0,3.0,0,5.0,5.0,3.0,1.0,PFD
1,2,0,3.0,5.0,3,0,3.0,5.0,3.0,0,...,0,3.0,5.0,3.0,0,3.0,5.0,3.0,1.0,PFD
2,3,0,5.0,5.0,3,0,3.0,5.0,3.0,0,...,0,5.0,5.0,3.0,0,5.0,5.0,3.0,33.0,PFDD
3,4,0,5.0,5.0,3,0,3.0,5.0,3.0,0,...,0,5.0,5.0,3.0,0,3.0,5.0,3.0,33.0,PFDD
4,5,0,5.0,5.0,3,0,3.0,5.0,3.0,0,...,0,5.0,5.0,3.0,0,5.0,5.0,3.0,1.0,PFD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158,164,0,3.0,5.0,3,0,5.0,5.0,3.0,0,...,0,5.0,5.0,3.0,0,5.0,5.0,3.0,18.0,PFD
159,165,0,5.0,5.0,3,0,3.0,5.0,3.0,0,...,0,5.0,5.0,3.0,0,3.0,5.0,3.0,28.0,PFDD
160,166,0,3.0,5.0,3,0,5.0,5.0,3.0,0,...,0,5.0,5.0,3.0,0,5.0,5.0,3.0,7.0,PFD
161,167,0,5.0,5.0,3,0,5.0,5.0,3.0,0,...,0,5.0,5.0,3.0,0,5.0,5.0,3.0,7.0,PFD


In [92]:
subject_results_df.Refined_Papilledema.value_counts()

Refined_Papilledema
3    161
5      1
Name: count, dtype: int64

In [93]:
list(subject_results_df)

['SubjectId',
 'Preop_Papilledema',
 'Index_Papilledema',
 'New_Papilledema',
 'Refined_Papilledema',
 'Preop_Nystagmus',
 'Index_Nystagmus',
 'New_Nystagmus',
 'Refined_Nystagmus',
 'Preop_DisconjugateGaze',
 'Index_DisconjugateGaze',
 'New_DisconjugateGaze',
 'Refined_DisconjugateGaze',
 'Preop_ExtraocularPalsies',
 'Index_DocExtraOcPalsies',
 'New_ExtraocularPalsies',
 'Refined_DocExtraOcPalsies',
 'Preop_FacialWeakness',
 'Index_FacialWeakness',
 'New_FacialWeakness',
 'Refined_FacialWeakness',
 'Preop_DisturbanceInFacialSensation',
 'Index_DisturbanceFacSen',
 'New_DisturbanceInFacialSensation',
 'Refined_DisturbanceFacSen',
 'Preop_HearingLoss',
 'Index_HearingLoss',
 'New_HearingLoss',
 'Refined_HearingLoss',
 'Preop_Hoarseness',
 'Index_Hoarseness',
 'New_Hoarseness',
 'Refined_Hoarseness',
 'Preop_TongueDeviation',
 'Index_TongueDeviation',
 'New_TongueDeviation',
 'Refined_TongueDeviation',
 'Preop_WeakShrug',
 'Index_WeakShrug',
 'New_WeakShrug',
 'Refined_WeakShrug',
 'Preo

In [94]:
for col in subject_results_df.columns:
    if col.startswith('Refined_'):
        print(f"{col} value counts:")
        print(subject_results_df[col].value_counts(dropna=False))
        print("-" * 40)

Refined_Papilledema value counts:
Refined_Papilledema
3    161
5      1
Name: count, dtype: int64
----------------------------------------
Refined_Nystagmus value counts:
Refined_Nystagmus
3.0    155
5.0      3
1.0      3
4.0      1
Name: count, dtype: int64
----------------------------------------
Refined_DisconjugateGaze value counts:
Refined_DisconjugateGaze
3.0    160
1.0      1
4.0      1
Name: count, dtype: int64
----------------------------------------
Refined_DocExtraOcPalsies value counts:
Refined_DocExtraOcPalsies
3.0    159
1.0      3
Name: count, dtype: int64
----------------------------------------
Refined_FacialWeakness value counts:
Refined_FacialWeakness
3.0    162
Name: count, dtype: int64
----------------------------------------
Refined_DisturbanceFacSen value counts:
Refined_DisturbanceFacSen
3.0    161
1.0      1
Name: count, dtype: int64
----------------------------------------
Refined_HearingLoss value counts:
Refined_HearingLoss
3.0    160
5.0      1
1.0      1
N

In [95]:
# convert to Limonadi Scoring
def map_to_limonadi(value):
    if value == 1:
        return 2
    elif value == 2:
        return 1
    elif value == 3:
        return 0
    elif value == 4:
        return -1
    elif value == 5:
        return 5

In [96]:
# Apply map_to_limonadi to all columns starting with 'Refined_' and create new 'Limonadi_' columns
for col in subject_results_df.columns:
    if col.startswith('Refined_'):
        limonadi_col = 'Limonadi_' + col.replace('Refined_', '')
        subject_results_df[limonadi_col] = subject_results_df[col].apply(map_to_limonadi)

In [97]:
list(subject_results_df)

['SubjectId',
 'Preop_Papilledema',
 'Index_Papilledema',
 'New_Papilledema',
 'Refined_Papilledema',
 'Preop_Nystagmus',
 'Index_Nystagmus',
 'New_Nystagmus',
 'Refined_Nystagmus',
 'Preop_DisconjugateGaze',
 'Index_DisconjugateGaze',
 'New_DisconjugateGaze',
 'Refined_DisconjugateGaze',
 'Preop_ExtraocularPalsies',
 'Index_DocExtraOcPalsies',
 'New_ExtraocularPalsies',
 'Refined_DocExtraOcPalsies',
 'Preop_FacialWeakness',
 'Index_FacialWeakness',
 'New_FacialWeakness',
 'Refined_FacialWeakness',
 'Preop_DisturbanceInFacialSensation',
 'Index_DisturbanceFacSen',
 'New_DisturbanceInFacialSensation',
 'Refined_DisturbanceFacSen',
 'Preop_HearingLoss',
 'Index_HearingLoss',
 'New_HearingLoss',
 'Refined_HearingLoss',
 'Preop_Hoarseness',
 'Index_Hoarseness',
 'New_Hoarseness',
 'Refined_Hoarseness',
 'Preop_TongueDeviation',
 'Index_TongueDeviation',
 'New_TongueDeviation',
 'Refined_TongueDeviation',
 'Preop_WeakShrug',
 'Index_WeakShrug',
 'New_WeakShrug',
 'Refined_WeakShrug',
 'Preo

In [98]:
len(list(subject_results_df))

163

In [99]:
new_order =[
 'SubjectId',
 'Preop_Papilledema',
 'Index_Papilledema',
 'New_Papilledema',
 'Refined_Papilledema',
 'Limonadi_Papilledema',
 'Preop_Nystagmus',
 'Index_Nystagmus',
 'New_Nystagmus',
 'Refined_Nystagmus',
 'Limonadi_Nystagmus',
 'Preop_DisconjugateGaze',
 'Index_DisconjugateGaze',
 'New_DisconjugateGaze',
 'Refined_DisconjugateGaze',
 'Limonadi_DisconjugateGaze',
 'Preop_ExtraocularPalsies',
 'New_ExtraocularPalsies',
 'Index_DocExtraOcPalsies',
 'Refined_DocExtraOcPalsies',
 'Limonadi_DocExtraOcPalsies',
 'Preop_FacialWeakness',
 'Index_FacialWeakness',
 'New_FacialWeakness',
 'Refined_FacialWeakness',
 'Limonadi_FacialWeakness',
 'Preop_DisturbanceInFacialSensation',
 'New_DisturbanceInFacialSensation',
 'Index_DisturbanceFacSen',
 'Refined_DisturbanceFacSen',
 'Limonadi_DisturbanceFacSen',
 'Preop_HearingLoss',
 'Index_HearingLoss',
 'New_HearingLoss',
 'Refined_HearingLoss',
 'Limonadi_HearingLoss',
 'Preop_Hoarseness',
 'Index_Hoarseness',
 'New_Hoarseness',
 'Refined_Hoarseness',
 'Limonadi_Hoarseness',
 'Preop_TongueDeviation',
 'Index_TongueDeviation',
 'New_TongueDeviation',
 'Refined_TongueDeviation',
 'Limonadi_TongueDeviation',
 'Preop_WeakShrug',
 'Index_WeakShrug',
 'New_WeakShrug',
 'Refined_WeakShrug',
 'Limonadi_WeakShrug',
 'Preop_CombinedUpperExtremityWeak',
 'Index_CombinedUpperExtremityStrength',
 'New_CombinedUpperExtremityStrength',
 'Refined_CombinedUpperExtremityStrength',
 'Limonadi_CombinedUpperExtremityStrength',
 'Preop_LeftUpperExtremityWeak',
 'Index_LeftUpperExtremityStrength',
 'New_LeftUpperExtremityStrength',
 'Refined_LeftUpperExtremityStrength',
 'Limonadi_LeftUpperExtremityStrength',
 'Preop_RightUpperExtremityWeak',
 'Index_RightUpperExtremityStrength',
 'New_RightUpperExtremityStrength',
 'Refined_RightUpperExtremityStrength',
 'Limonadi_RightUpperExtremityStrength',
 'Preop_CombinedLowerExtremityWeak',
 'Index_CombinedLowerExtremityStrength',
 'New_CombinedLowerExtremityStrength',
 'Refined_CombinedLowerExtremityStrength',
 'Limonadi_CombinedLowerExtremityStrength',
 'Preop_LeftLowerExtremityWeak',
 'Index_LeftLowerExtremityStrength',
 'New_LeftLowerExtremityStrength',
 'Refined_LeftLowerExtremityStrength',
 'Limonadi_LeftLowerExtremityStrength',
 'Preop_RightLowerExtremityWeak',
 'Index_RightLowerExtremityStrength',
 'New_RightLowerExtremityStrength',
 'Refined_RightLowerExtremityStrength',
 'Limonadi_RightLowerExtremityStrength',
 'Preop_LighttouchEntireBody',
 'Index_DeficitToLightTouchEntireBody',
 'New_LighttouchEntireBody',
 'Refined_DeficitToLightTouchEntireBody',
 'Limonadi_DeficitToLightTouchEntireBody',
 'Preop_CombinedUpperDLTExtremities',
 'Index_CombinedUpperDLTExtremities',
 'New_CombinedUpperDLTExtremities',
 'Refined_CombinedUpperDLTExtremities',
 'Limonadi_CombinedUpperDLTExtremities',
 'Preop_DLTLeftUpperExtremities',
 'Index_LightTouchSensoryDeficitLUE',
 'New_DLTLeftUpperExtremities',
 'Refined_LightTouchSensoryDeficitLUE',
 'Limonadi_LightTouchSensoryDeficitLUE',
 'Preop_DLTRightUpperExtremities',
 'Index_LightTouchSensoryDeficitRUE',
 'New_DLTRightUpperExtremities',
 'Refined_LightTouchSensoryDeficitRUE',
 'Limonadi_LightTouchSensoryDeficitRUE',
 'Preop_CombinedLowerDLTExtremities',
 'Index_CombinedLowerDLTExtremities',
 'New_CombinedLowerDLTExtremities',
 'Refined_CombinedLowerDLTExtremities',
 'Limonadi_CombinedLowerDLTExtremities',
 'Preop_DLTLeftLowerExtremities',
 'Index_LightTouchSensoryDeficitLLE',
 'New_DLTLeftLowerExtremities',
 'Refined_LightTouchSensoryDeficitLLE',
 'Limonadi_LightTouchSensoryDeficitLLE',
 'Preop_DLTRightLowerExtremities',
 'Index_LightTouchSensoryDeficitRLE',
 'New_DLTRightLowerExtremities',
 'Refined_LightTouchSensoryDeficitRLE',
 'Limonadi_LightTouchSensoryDeficitRLE',
 'Preop_DeficitToPinprickUpperLeft',
 'Index_LUEDeficitPinprick',
 'New_DeficitToPinprickUpperLeft',
 'Refined_LUEDeficitPinprick',
 'Limonadi_LUEDeficitPinprick',
 'Preop_DeficitToPinprickUpperRight',
 'Index_RUEDeficitPinprick',
 'New_DeficitToPinprickUpperRight',
 'Refined_RUEDeficitPinprick',
 'Limonadi_RUEDeficitPinprick',
 'Preop_DeficittoPinprickLowerLeft',
 'Index_LLEDeficitPinprick',
 'New_DeficittoPinprickLowerLeft',
 'Refined_LLEDeficitPinprick',
 'Limonadi_LLEDeficitPinprick',
 'Preop_DeficittoPinprickLowerRight',
 'Index_RLEDeficitPinprick',
 'New_DeficittoPinprickLowerRight',
 'Refined_RLEDeficitPinprick',
 'Limonadi_RLEDeficitPinprick',
 'Preop_LightTouchTorso',
 'Index_DeficitLightTouchTorso',
 'New_LightTouchTorso',
 'Refined_DeficitLightTouchTorso',
 'Limonadi_DeficitLightTouchTorso',
 'Preop_LightTouchSaddle',
 'Index_DeficitLIghtTouchSaddle',
 'New_LightTouchSaddle',
 'Refined_DeficitLIghtTouchSaddle',
 'Limonadi_DeficitLIghtTouchSaddle',
 'Preop_DeepTendonReflexes',
 'Index_DeepTendonReflexes',
 'New_DeepTendonReflexes',
 'Refined_DeepTendonReflexes',
 'Limonadi_DeepTendonReflexes',
 'Preop_AnkleClonus',
 'Index_FUAnkleClonus',
 'New_AnkleClonus',
 'Refined_FUAnkleClonus',
 'Limonadi_FUAnkleClonus',
 'Preop_GaitInstability',
 'New_GaitInstability',
 'Index_GaitStatus',
 'Refined_GaitStatus',
 'Limonadi_GaitStatus',
 'Site',
 'Allocation'
]


In [100]:
len(new_order)

163

In [101]:
subject_results_df = subject_results_df[new_order]

In [102]:
subject_results_df

,SubjectId,Preop_Papilledema,Index_Papilledema,New_Papilledema,Refined_Papilledema,Limonadi_Papilledema,Preop_Nystagmus,Index_Nystagmus,New_Nystagmus,Refined_Nystagmus,...,New_AnkleClonus,Refined_FUAnkleClonus,Limonadi_FUAnkleClonus,Preop_GaitInstability,New_GaitInstability,Index_GaitStatus,Refined_GaitStatus,Limonadi_GaitStatus,Site,Allocation
0,1,0,3.0,5.0,3,0,0,5.0,5.0,3.0,...,5.0,3.0,0,0,5.0,5.0,3.0,0,1.0,PFD
1,2,0,3.0,5.0,3,0,0,3.0,5.0,3.0,...,5.0,3.0,0,0,5.0,3.0,3.0,0,1.0,PFD
2,3,0,5.0,5.0,3,0,0,3.0,5.0,3.0,...,5.0,3.0,0,0,5.0,5.0,3.0,0,33.0,PFDD
3,4,0,5.0,5.0,3,0,0,3.0,5.0,3.0,...,5.0,3.0,0,0,5.0,3.0,3.0,0,33.0,PFDD
4,5,0,5.0,5.0,3,0,0,3.0,5.0,3.0,...,5.0,3.0,0,0,5.0,5.0,3.0,0,1.0,PFD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158,164,0,3.0,5.0,3,0,0,5.0,5.0,3.0,...,5.0,3.0,0,0,5.0,5.0,3.0,0,18.0,PFD
159,165,0,5.0,5.0,3,0,0,3.0,5.0,3.0,...,5.0,3.0,0,0,5.0,3.0,3.0,0,28.0,PFDD
160,166,0,3.0,5.0,3,0,0,5.0,5.0,3.0,...,5.0,3.0,0,0,5.0,5.0,3.0,0,7.0,PFD
161,167,0,5.0,5.0,3,0,0,5.0,5.0,3.0,...,5.0,3.0,0,0,5.0,5.0,3.0,0,7.0,PFD


In [103]:
import numpy as np

nan_counts = subject_results_df.isna().sum()
print("Columns with NaN values:")
print(nan_counts[nan_counts > 0])

Columns with NaN values:
Series([], dtype: int64)


In [104]:
pd.DataFrame(nan_counts[nan_counts > 0]).reset_index()['index'].unique()

array([], dtype=object)

In [105]:
# Replace 'Limonadi_' with 'Refined_' in interested_features
interested_features = ['Limonadi_DocExtraOcPalsies',
       'Limonadi_CombinedUpperExtremityStrength',
       'Limonadi_CombinedLowerExtremityStrength',
       'Limonadi_CombinedUpperDLTExtremities',
       'Limonadi_LightTouchSensoryDeficitLUE',
       'Limonadi_LightTouchSensoryDeficitRUE',
       'Limonadi_CombinedLowerDLTExtremities',
       'Limonadi_LightTouchSensoryDeficitLLE',
       'Limonadi_LightTouchSensoryDeficitRLE']
refined_features = [f.replace('Limonadi_', 'Refined_') for f in interested_features]
print(refined_features)

['Refined_DocExtraOcPalsies', 'Refined_CombinedUpperExtremityStrength', 'Refined_CombinedLowerExtremityStrength', 'Refined_CombinedUpperDLTExtremities', 'Refined_LightTouchSensoryDeficitLUE', 'Refined_LightTouchSensoryDeficitRUE', 'Refined_CombinedLowerDLTExtremities', 'Refined_LightTouchSensoryDeficitLLE', 'Refined_LightTouchSensoryDeficitRLE']


In [106]:
refined_features

['Refined_DocExtraOcPalsies',
 'Refined_CombinedUpperExtremityStrength',
 'Refined_CombinedLowerExtremityStrength',
 'Refined_CombinedUpperDLTExtremities',
 'Refined_LightTouchSensoryDeficitLUE',
 'Refined_LightTouchSensoryDeficitRUE',
 'Refined_CombinedLowerDLTExtremities',
 'Refined_LightTouchSensoryDeficitLLE',
 'Refined_LightTouchSensoryDeficitRLE']

In [107]:
subject_results_df['Index_DocExtraOcPalsies'].value_counts()

Index_DocExtraOcPalsies
3.0    123
5.0     36
1.0      3
Name: count, dtype: int64

In [108]:
subject_results_df['Refined_DocExtraOcPalsies'].value_counts()

Refined_DocExtraOcPalsies
3.0    159
1.0      3
Name: count, dtype: int64

In [109]:
for col in refined_features:
    print(subject_results_df[col].value_counts())

Refined_DocExtraOcPalsies
3.0    159
1.0      3
Name: count, dtype: int64
Refined_CombinedUpperExtremityStrength
3    158
1      3
5      1
Name: count, dtype: int64
Refined_CombinedLowerExtremityStrength
3    161
4      1
Name: count, dtype: int64
Refined_CombinedUpperDLTExtremities
3    156
1      2
5      2
4      1
2      1
Name: count, dtype: int64
Refined_LightTouchSensoryDeficitLUE
3.0    160
1.0      1
2.0      1
Name: count, dtype: int64
Refined_LightTouchSensoryDeficitRUE
3.0    158
1.0      2
4.0      1
2.0      1
Name: count, dtype: int64
Refined_CombinedLowerDLTExtremities
3    161
4      1
Name: count, dtype: int64
Refined_LightTouchSensoryDeficitLLE
3.0    162
Name: count, dtype: int64
Refined_LightTouchSensoryDeficitRLE
3.0    162
Name: count, dtype: int64


In [110]:


# Split into two groups
pfd_df = subject_results_df[subject_results_df['Allocation'] == 'PFD'].copy()
pfdd_df = subject_results_df[subject_results_df['Allocation'] == 'PFDD'].copy()
pfd_df = pfd_df.dropna(subset=['SubjectId'])
pfdd_df = pfdd_df.dropna(subset=['SubjectId'])

# Preview
print("PFD group:", pfd_df.shape)
print("PFDD group:", pfdd_df.shape)


PFD group: (84, 163)
PFDD group: (78, 163)


In [111]:
list(pfd_df)

['SubjectId',
 'Preop_Papilledema',
 'Index_Papilledema',
 'New_Papilledema',
 'Refined_Papilledema',
 'Limonadi_Papilledema',
 'Preop_Nystagmus',
 'Index_Nystagmus',
 'New_Nystagmus',
 'Refined_Nystagmus',
 'Limonadi_Nystagmus',
 'Preop_DisconjugateGaze',
 'Index_DisconjugateGaze',
 'New_DisconjugateGaze',
 'Refined_DisconjugateGaze',
 'Limonadi_DisconjugateGaze',
 'Preop_ExtraocularPalsies',
 'New_ExtraocularPalsies',
 'Index_DocExtraOcPalsies',
 'Refined_DocExtraOcPalsies',
 'Limonadi_DocExtraOcPalsies',
 'Preop_FacialWeakness',
 'Index_FacialWeakness',
 'New_FacialWeakness',
 'Refined_FacialWeakness',
 'Limonadi_FacialWeakness',
 'Preop_DisturbanceInFacialSensation',
 'New_DisturbanceInFacialSensation',
 'Index_DisturbanceFacSen',
 'Refined_DisturbanceFacSen',
 'Limonadi_DisturbanceFacSen',
 'Preop_HearingLoss',
 'Index_HearingLoss',
 'New_HearingLoss',
 'Refined_HearingLoss',
 'Limonadi_HearingLoss',
 'Preop_Hoarseness',
 'Index_Hoarseness',
 'New_Hoarseness',
 'Refined_Hoarseness

In [112]:
list(pfdd_df)

['SubjectId',
 'Preop_Papilledema',
 'Index_Papilledema',
 'New_Papilledema',
 'Refined_Papilledema',
 'Limonadi_Papilledema',
 'Preop_Nystagmus',
 'Index_Nystagmus',
 'New_Nystagmus',
 'Refined_Nystagmus',
 'Limonadi_Nystagmus',
 'Preop_DisconjugateGaze',
 'Index_DisconjugateGaze',
 'New_DisconjugateGaze',
 'Refined_DisconjugateGaze',
 'Limonadi_DisconjugateGaze',
 'Preop_ExtraocularPalsies',
 'New_ExtraocularPalsies',
 'Index_DocExtraOcPalsies',
 'Refined_DocExtraOcPalsies',
 'Limonadi_DocExtraOcPalsies',
 'Preop_FacialWeakness',
 'Index_FacialWeakness',
 'New_FacialWeakness',
 'Refined_FacialWeakness',
 'Limonadi_FacialWeakness',
 'Preop_DisturbanceInFacialSensation',
 'New_DisturbanceInFacialSensation',
 'Index_DisturbanceFacSen',
 'Refined_DisturbanceFacSen',
 'Limonadi_DisturbanceFacSen',
 'Preop_HearingLoss',
 'Index_HearingLoss',
 'New_HearingLoss',
 'Refined_HearingLoss',
 'Limonadi_HearingLoss',
 'Preop_Hoarseness',
 'Index_Hoarseness',
 'New_Hoarseness',
 'Refined_Hoarseness

In [113]:
# Define mapping explicitly for this dataset
mapping = {
    "Preop_Papilledema": "Index_Papilledema",
    "Preop_Nystagmus": "Index_Nystagmus",
    "Preop_DisconjugateGaze": "Index_DisconjugateGaze",
    "Preop_ExtraocularPalsies": "Index_DocExtraOcPalsies",
    "Preop_FacialWeakness": "Index_FacialWeakness",
    "Preop_DisturbanceInFacialSensation": "Index_DisturbanceFacSen",
    "Preop_HearingLoss": "Index_HearingLoss",
    "Preop_Hoarseness": "Index_Hoarseness",
    "Preop_TongueDeviation": "Index_TongueDeviation",
    "Preop_WeakShrug": "Index_WeakShrug",
    "Preop_CombinedUpperExtremityWeak": "Index_CombinedUpperExtremityStrength",
    "Preop_CombinedLowerExtremityWeak": "Index_CombinedLowerExtremityStrength",
    "Preop_LeftUpperExtremityWeak": "Index_LeftUpperExtremityStrength",
    "Preop_RightUpperExtremityWeak": "Index_RightUpperExtremityStrength",
    "Preop_LeftLowerExtremityWeak": "Index_LeftLowerExtremityStrength",
    "Preop_RightLowerExtremityWeak": "Index_RightLowerExtremityStrength",
    "Preop_LighttouchEntireBody": "Index_DeficitToLightTouchEntireBody",
    "Preop_CombinedUpperDLTExtremities": "Index_CombinedUpperDLTExtremities",
    "Preop_DLTLeftUpperExtremities": "Index_LightTouchSensoryDeficitLUE",
    "Preop_DLTRightUpperExtremities": "Index_LightTouchSensoryDeficitRUE",
    "Preop_CombinedLowerDLTExtremities": "Index_CombinedLowerDLTExtremities",
    "Preop_DLTLeftLowerExtremities": "Index_LightTouchSensoryDeficitLLE",
    "Preop_DLTRightLowerExtremities": "Index_LightTouchSensoryDeficitRLE",
    "Preop_DeficitToPinprickUpperLeft": "Index_LUEDeficitPinprick",
    "Preop_DeficitToPinprickUpperRight": "Index_RUEDeficitPinprick",
    "Preop_DeficittoPinprickLowerLeft": "Index_LLEDeficitPinprick",
    "Preop_DeficittoPinprickLowerRight": "Index_RLEDeficitPinprick",
    "Preop_LightTouchTorso": "Index_DeficitLightTouchTorso",
    "Preop_LightTouchSaddle": "Index_DeficitLIghtTouchSaddle",
    "Preop_DeepTendonReflexes": "Index_DeepTendonReflexes",
    "Preop_AnkleClonus": "Index_FUAnkleClonus",
    "Preop_GaitInstability": "Index_GaitStatus",
}

# Apply your rule
for preop_col, index_col in mapping.items():
    subject_results_df.loc[(subject_results_df[preop_col] == 0) & (subject_results_df[index_col] == 5), index_col] = 0


In [114]:
subject_results_df

,SubjectId,Preop_Papilledema,Index_Papilledema,New_Papilledema,Refined_Papilledema,Limonadi_Papilledema,Preop_Nystagmus,Index_Nystagmus,New_Nystagmus,Refined_Nystagmus,...,New_AnkleClonus,Refined_FUAnkleClonus,Limonadi_FUAnkleClonus,Preop_GaitInstability,New_GaitInstability,Index_GaitStatus,Refined_GaitStatus,Limonadi_GaitStatus,Site,Allocation
0,1,0,3.0,5.0,3,0,0,0.0,5.0,3.0,...,5.0,3.0,0,0,5.0,0.0,3.0,0,1.0,PFD
1,2,0,3.0,5.0,3,0,0,3.0,5.0,3.0,...,5.0,3.0,0,0,5.0,3.0,3.0,0,1.0,PFD
2,3,0,0.0,5.0,3,0,0,3.0,5.0,3.0,...,5.0,3.0,0,0,5.0,0.0,3.0,0,33.0,PFDD
3,4,0,0.0,5.0,3,0,0,3.0,5.0,3.0,...,5.0,3.0,0,0,5.0,3.0,3.0,0,33.0,PFDD
4,5,0,0.0,5.0,3,0,0,3.0,5.0,3.0,...,5.0,3.0,0,0,5.0,0.0,3.0,0,1.0,PFD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158,164,0,3.0,5.0,3,0,0,0.0,5.0,3.0,...,5.0,3.0,0,0,5.0,0.0,3.0,0,18.0,PFD
159,165,0,0.0,5.0,3,0,0,3.0,5.0,3.0,...,5.0,3.0,0,0,5.0,3.0,3.0,0,28.0,PFDD
160,166,0,3.0,5.0,3,0,0,0.0,5.0,3.0,...,5.0,3.0,0,0,5.0,0.0,3.0,0,7.0,PFD
161,167,0,0.0,5.0,3,0,0,0.0,5.0,3.0,...,5.0,3.0,0,0,5.0,0.0,3.0,0,7.0,PFD


In [115]:
subject_results_df[subject_results_df['SubjectId'] == 9]

,SubjectId,Preop_Papilledema,Index_Papilledema,New_Papilledema,Refined_Papilledema,Limonadi_Papilledema,Preop_Nystagmus,Index_Nystagmus,New_Nystagmus,Refined_Nystagmus,...,New_AnkleClonus,Refined_FUAnkleClonus,Limonadi_FUAnkleClonus,Preop_GaitInstability,New_GaitInstability,Index_GaitStatus,Refined_GaitStatus,Limonadi_GaitStatus,Site,Allocation
8,9,0,0.0,5.0,3,0,0,3.0,5.0,3.0,...,5.0,3.0,0,0,5.0,3.0,3.0,0,47.0,PFDD


In [116]:
pfd_df.to_csv('pfd_result_table1_neurological_exam_0930.csv',index=False)
pfdd_df.to_csv('pfdd_result_table1_neurological_exam_0930.csv',index=False)

In [117]:
# # Function to generate summary table for a given group
# def summarize_symptoms(df, label='Group'):
#     all_summaries = []

#     for pre_symptom, post_symptom in neuro_sign_map.items():
# #        print(post_symptom)
#         # filter with preoperatove data
#         pre_symptom = pre_symptom+'_preop'
#         subset = df[df[pre_symptom] == False]
#         baseline_count = subset['SubjectId'].nunique()

#         temp = subset[['SubjectId', post_symptom]].fillna(0)
#         temp = temp.rename(columns={post_symptom: 'Outcome'})
#         temp['Outcome'] = temp['Outcome']
#         temp['Symptom'] = pre_symptom.split('_')[0]
# #        print(temp.Outcome.value_counts())
#         count_df = temp.groupby('Outcome')['SubjectId'].nunique().reset_index()
#         count_df['Symptom'] = pre_symptom.split('_')[0]
#         count_df = count_df.rename(columns={'SubjectId': 'N_Patients'})
#         count_pivot = count_df.pivot(index='Symptom', columns='Outcome', values='N_Patients')
#         count_pivot['NotPresentBaseline'] = baseline_count
# #        print(count_pivot)
#         all_summaries.append(count_pivot)
#     # 4. Rename outcome columns
# # ---- Combine Left and Right Extremities ----

#     # Define logical groupings of left/right into higher-level symptoms
#     combo_groups = {
#         'CombinedUpperExtremityWeak': ['LeftUpperExtremityWeak', 'RightUpperExtremityWeak'],
#         'CombinedLowerExtremityWeak': ['LeftLowerExtremityWeak', 'RightLowerExtremityWeak'],
#         'CombinedUpperDLTExtremity': ['DLTLeftUpperExtremities', 'DLTRightUpperExtremities'],
#         'CombinedLowerDLTExtremity': ['DLTLeftLowerExtremities', 'DLTRightLowerExtremities'],
#     }

#     for new_symptom, sub_symptoms in combo_groups.items():
#         # Get preop columns
#         preop_cols = ["Preop_"s for s in sub_symptoms]

#         # Subjects who had all false
#         patients_union = set(df.loc[df[preop_cols].eq(False).all(axis=1), 'SubjectId'])

#         baseline_count = len(patients_union)

#         # Get corresponding post-op column names from neuro_sign_map
#         postop_cols = [neuro_sign_map[s] for s in sub_symptoms]

#         # Get the actual data
#         subset = df[df['SubjectId'].isin(patients_union)][['SubjectId'] + postop_cols].fillna(0)

#         # Use max value across post-op indicators
#         subset['Outcome'] = subset[postop_cols].max(axis=1)
#         subset['Symptom'] = new_symptom
#         print(subset)
#         print(subset.columns)
#         # Count distinct subjects per outcome category
#         count_df = subset.groupby('Outcome')['SubjectId'].nunique().reset_index()

#         count_df['Symptom'] = new_symptom
#         count_df = count_df.rename(columns={'SubjectId': 'N_Patients'})
#         count_pivot = count_df.pivot(index='Symptom', columns='Outcome', values='N_Patients').fillna(0)
#         count_pivot['NotPresentBaseline'] = baseline_count

#         all_summaries.append(count_pivot)

#     outcome_labels = {
#         1: 'Resolved',
#         2: 'Improved',
#         3: 'Stable',
#         4: 'Worse',
#         0: 'Unknown',
#         'Nan in the datasheets':'Nan in the datasheets',
#         '2': 'Improved',
#         '0':'Unknown',
#         '1':'Resolved',
#         '3':'Stable',
#         '4':'Worse'
#     }

#     # 5. Reorder columns (only include if present)
#     desired_order = [
#         'NotPresentBaseline', 'Resolved', 'Improved', 'Stable', 'Worse',
#         'Unknown','Nan in the datasheets'
#     ]

#     # Combine and rename
#     summary_df = pd.concat(all_summaries).fillna(0)

# #     summary_df = summary_df.replace({
# #     '1': 1,
# #     '2': 2,
# #     '3': 3,
# #         '4':4,
# #         '0':0
# # })
#     summary_df = summary_df.rename(columns=outcome_labels)

#     # Reorder
#     summary_df = summary_df[[col for col in desired_order if col in summary_df.columns]]
#     summary_df = summary_df.reset_index()
#     summary_df['Group'] = label

#     return summary_df,subset


# # Apply separately to each group
# pfd_summary,pfd_subset = summarize_symptoms(pfd_df, label='PFD')
# print('======================================================split betwwen pfd and pfdd============================================================================================')
# pfdd_summary,pfdd_subset = summarize_symptoms(pfdd_df, label='PFDD')


# pfd_summary
